# Agent02 · Code đầy đủ và kết quả

Notebook là nơi đọc và sửa toàn bộ implementation dưới đây: dữ liệu, mô hình, loss, optimizer, train/đánh giá. Không cần mở file `.py` để xem thuật toán. Các bản `.py` cũ giữ nguyên để bảo toàn backend/artifact.

**Cách đọc:** khởi tạo → code từng hàm/lớp → thực hành theo dataset → kết quả. `%%inline_module` chỉ chạy code có ngay trong cell ở namespace riêng, không nạp implementation từ `.py`. Mỗi framework dùng kernel WSL riêng; code của cả hai vẫn hiện trong notebook. Mặc định không train/TEST lại.

In [1]:
PROJECT_NAME = 'agent02'
from pathlib import Path
import sys, os, types, __future__
from IPython.core.magic import register_cell_magic
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ASSIGN_REORGANIZATION_EXECUTION_PLAN.md").exists())
sys.path.insert(0, str(REPO / "_shared"))
from assignment_runtime import activate
ACTIVE_FRAMEWORK = "keras" if ".venv-keras" in sys.executable else "torch"
_context = activate(PROJECT_NAME, ACTIVE_FRAMEWORK)
LOGICAL_ROOT = Path(_context["logical_root"])
RUN_EXPERIMENTS = False
RUN_TRAINING = False
RUN_TEST = False
# Kết quả của lượt chạy mới nằm ngoài notebooks/, không ghi vào artifact gốc.
import assignment_runtime as _paths
_paths._WRITE_CONTEXT["output"] = REPO / PROJECT_NAME / "artifacts/notebook_runs/inline_v1"
_paths._WRITE_CONTEXT["overlay"] = False
INLINE_MODULES = {}

def _notebook_package(name):
    if not name:
        return None
    if name not in INLINE_MODULES:
        parent, _, child = name.rpartition(".")
        package = _notebook_package(parent)
        module = types.ModuleType(name)
        module.__package__ = parent
        module.__path__ = []  # Không tìm implementation khác trong file .py.
        module.__file__ = str(LOGICAL_ROOT / (name.replace(".", "/") + "/__init__.py"))
        module.__dict__.update(ACTIVE_FRAMEWORK=ACTIVE_FRAMEWORK, RUN_TRAINING=False, RUN_TEST=False)
        INLINE_MODULES[name] = module
        sys.modules[name] = module
        if package is not None:
            setattr(package, child, module)
    return INLINE_MODULES[name]

@register_cell_magic
def inline_module(line, cell):
    """Chạy đúng code trong cell ở namespace riêng; không đọc code từ .py."""
    name, relative, scope = line.split()
    if scope != "both" and scope != ACTIVE_FRAMEWORK:
        return
    parent, _, child = name.rpartition(".")
    _notebook_package(parent)
    module = _notebook_package(name)
    module.__file__ = str(LOGICAL_ROOT / relative)
    module.__package__ = name if relative.endswith("/__init__.py") else parent
    compiled = compile(cell, "<notebook>/" + name, "exec", flags=__future__.annotations.compiler_flag, dont_inherit=True)
    exec(compiled, module.__dict__)
    module.__notebook_source__ = True

print({"bài": PROJECT_NAME, "kernel": ACTIVE_FRAMEWORK, "train": RUN_TRAINING, "TEST": RUN_TEST})


{'bài': 'agent02', 'kernel': 'torch', 'train': False, 'TEST': False}


### `agent02_assignment.code.controlled`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [2]:
%%inline_module agent02_assignment.code.controlled agent02_assignment/code/controlled.py both
"""Teaching market and tabular policies; CONTROLLED v1 uses separate random streams."""

from dataclasses import dataclass

from numbers import Integral

import numpy as np



#### integer

In [3]:
%%inline_module agent02_assignment.code.controlled agent02_assignment/code/controlled.py both
def integer(value, name, minimum=0):
    if isinstance(value, bool) or not isinstance(value, Integral) or value < minimum:
        raise ValueError(f'{name} must be an integer >= {minimum}')
    return int(value)



#### binary

In [4]:
%%inline_module agent02_assignment.code.controlled agent02_assignment/code/controlled.py both
def binary(value, name='binary value'):
    result = integer(value, name)
    if result > 1:
        raise ValueError(f'{name} must be 0 or 1')
    return result



#### state_pair

In [5]:
%%inline_module agent02_assignment.code.controlled agent02_assignment/code/controlled.py both
def state_pair(state):
    if not isinstance(state, (tuple, list)) or len(state) != 2:
        raise ValueError('Local state must contain demand and competitor previous action')
    return tuple(binary(value) for value in state)



#### MarketConfig

In [6]:
%%inline_module agent02_assignment.code.controlled agent02_assignment/code/controlled.py both
@dataclass(frozen=True)
class MarketConfig:
    steps: int = 20
    price: float = 10.0
    cost: float = 5.0
    discount: float = .20
    customers_low: int = 10
    customers_high: int = 20
    kernel: tuple = ((.5, .5), (.5, .5))

    def __post_init__(self):
        for name in ('steps', 'customers_low', 'customers_high'):
            integer(getattr(self, name), name, 1)
        values = [self.price, self.cost, self.discount]
        if not np.isfinite(values).all() or self.price <= 0 or self.cost < 0 or not 0 <= self.discount <= 1:
            raise ValueError('Invalid price, cost or discount')
        kernel = np.asarray(self.kernel, dtype=float)
        if (kernel.shape != (2, 2) or not np.isfinite(kernel).all() or (kernel < 0).any()
                or not np.allclose(kernel.sum(axis=1), 1, rtol=0, atol=1e-12)):
            raise ValueError('Demand kernel must be a 2x2 stochastic matrix')
        object.__setattr__(self, 'kernel', tuple(tuple(float(x) for x in row) for row in kernel))



#### LearningConfig

In [7]:
%%inline_module agent02_assignment.code.controlled agent02_assignment/code/controlled.py both
@dataclass(frozen=True)
class LearningConfig:
    episodes: int = 5000
    alpha: float = .10
    gamma: float = .90
    epsilon_start: float = .20
    epsilon_min: float = .02
    epsilon_decay: float = .995

    def __post_init__(self):
        integer(self.episodes, 'episodes', 1)
        if (not np.isfinite([self.alpha, self.gamma, self.epsilon_start,
                             self.epsilon_min, self.epsilon_decay]).all()
                or not 0 < self.alpha <= 1 or not 0 <= self.gamma <= 1
                or not 0 <= self.epsilon_min <= self.epsilon_start <= 1
                or not 0 < self.epsilon_decay <= 1):
            raise ValueError('Invalid learning parameters')

    def epsilon_at(self, episode):
        integer(episode, 'episode')
        return max(self.epsilon_min, self.epsilon_start * self.epsilon_decay ** episode)



#### conversion

In [8]:
%%inline_module agent02_assignment.code.controlled agent02_assignment/code/controlled.py both
def conversion(demand, own_action, other_action):
    d, own, other = binary(demand), binary(own_action), binary(other_action)
    return float(np.clip(.30 + .25*d + .15*own - .10*other, .05, .95))



#### margin

In [9]:
%%inline_module agent02_assignment.code.controlled agent02_assignment/code/controlled.py both
def margin(config, action):
    return config.price * (1 - config.discount * binary(action)) - config.cost



#### Tape

In [10]:
%%inline_module agent02_assignment.code.controlled agent02_assignment/code/controlled.py both
@dataclass(frozen=True)
class Tape:
    """One episode of exogenous demand and independent customer uniforms, not sales."""
    demand: tuple
    uniforms: np.ndarray

    def __post_init__(self):
        demand = tuple(binary(d) for d in self.demand)
        uniforms = np.array(self.uniforms, dtype=np.float64, copy=True)
        if (len(demand) < 2 or uniforms.ndim != 3 or uniforms.shape[:2] != (len(demand)-1, 2)
                or uniforms.shape[2] < 1 or not np.isfinite(uniforms).all()
                or (uniforms < 0).any() or (uniforms >= 1).any()):
            raise ValueError('Malformed scenario tape')
        uniforms.flags.writeable = False
        object.__setattr__(self, 'demand', demand)
        object.__setattr__(self, 'uniforms', uniforms)

    @classmethod
    def generate(cls, config, scenario_seed, episode):
        integer(scenario_seed, 'scenario_seed')
        integer(episode, 'episode')
        demand_rng = np.random.default_rng(np.random.SeedSequence([scenario_seed, episode, 0]))
        sales_rng = np.random.default_rng(np.random.SeedSequence([scenario_seed, episode, 1]))
        demand = [int(demand_rng.integers(2))]
        for _ in range(config.steps):
            demand.append(int(demand_rng.random() >= config.kernel[demand[-1]][0]))
        uniforms = sales_rng.random((config.steps, 2, max(config.customers_low, config.customers_high)))
        return cls(tuple(demand), uniforms)



#### StepResult

In [11]:
%%inline_module agent02_assignment.code.controlled agent02_assignment/code/controlled.py both
@dataclass(frozen=True)
class StepResult:
    states: tuple
    rewards: tuple
    sales: tuple
    probabilities: tuple
    customers: int
    done: bool



#### MarketEnv

In [12]:
%%inline_module agent02_assignment.code.controlled agent02_assignment/code/controlled.py both
class MarketEnv:
    def __init__(self, config, seed):
        self.config = config
        integer(seed, 'seed')
        self.demand_rng = np.random.default_rng(np.random.SeedSequence([seed, 0]))
        self.sales_rngs = [np.random.default_rng(np.random.SeedSequence([seed, index])) for index in (1, 2)]
        self.demand, self.previous, self.elapsed = 0, (0, 0), 0
        self.ready, self.tape = False, None

    def states(self):
        return ((self.demand, self.previous[1]), (self.demand, self.previous[0]))

    def reset(self, tape=None):
        if tape is not None and tape.uniforms.shape != (
                self.config.steps, 2, max(self.config.customers_low, self.config.customers_high)):
            raise ValueError('Tape dimensions do not match market configuration')
        self.tape = tape
        self.demand = tape.demand[0] if tape is not None else int(self.demand_rng.integers(2))
        self.previous, self.elapsed, self.ready = (0, 0), 0, True
        return self.states()

    def step(self, actions):
        if not self.ready or self.elapsed >= self.config.steps:
            raise RuntimeError('Call reset before stepping or after the episode ends')
        if not isinstance(actions, (tuple, list)) or len(actions) != 2:
            raise ValueError('Exactly two simultaneous actions are required')
        actions = tuple(binary(action, 'action') for action in actions)
        customers = self.config.customers_high if self.demand else self.config.customers_low
        probabilities = tuple(conversion(self.demand, actions[i], actions[1-i]) for i in range(2))
        if self.tape is None:
            sales = tuple(int(self.sales_rngs[i].binomial(customers, probabilities[i])) for i in range(2))
            next_demand = int(self.demand_rng.random() >= self.config.kernel[self.demand][0])
        else:
            sales = tuple(int((self.tape.uniforms[self.elapsed, i, :customers] < probabilities[i]).sum())
                          for i in range(2))
            next_demand = self.tape.demand[self.elapsed+1]
        rewards = tuple(sales[i] * margin(self.config, actions[i]) for i in range(2))
        self.previous, self.demand = actions, next_demand
        self.elapsed += 1
        return StepResult(self.states(), rewards, sales, probabilities, customers,
                          self.elapsed == self.config.steps)



#### RuleAgent

In [13]:
%%inline_module agent02_assignment.code.controlled agent02_assignment/code/controlled.py both
class RuleAgent:
    def act(self, state):
        demand, competitor_previous = state_pair(state)
        return int(demand == 1 and competitor_previous == 1)



#### QAgent

In [14]:
%%inline_module agent02_assignment.code.controlled agent02_assignment/code/controlled.py both
class QAgent:
    def __init__(self, config, seed):
        self.config = config
        integer(seed, 'seed')
        self.rng = np.random.default_rng(seed)
        self.q = np.zeros((2, 2, 2), dtype=np.float64)

    def act(self, state, epsilon, *, explore=True):
        state = state_pair(state)
        if not np.isfinite(epsilon) or not 0 <= epsilon <= 1:
            raise ValueError('epsilon must be in [0,1]')
        values = self.q[state]
        if not np.isfinite(values).all():
            raise ValueError('Nonfinite Q values')
        if explore and self.rng.random() < epsilon:
            return int(self.rng.integers(2))
        # Same near-tie convention as the reference, made explicit in the protocol.
        best = np.flatnonzero(np.isclose(values, values.max(), rtol=1e-5, atol=1e-8))
        return int(self.rng.choice(best))

    def learn(self, state, action, reward, following, terminal):
        state, following, action = state_pair(state), state_pair(following), binary(action, 'action')
        if not np.isfinite(reward):
            raise ValueError('Nonfinite reward')
        target = reward if terminal else reward + self.config.gamma * float(self.q[following].max())
        key = (*state, action)
        self.q[key] += self.config.alpha * (target - self.q[key])
        if not np.isfinite(self.q[key]):
            raise ValueError('Nonfinite Q update')



### `agent02_assignment.code.training`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [15]:
%%inline_module agent02_assignment.code.training agent02_assignment/code/training.py both
"""Episode-boundary checkpoints record Q, logs and every independent RNG stream."""

import copy

import hashlib

import json

from dataclasses import asdict

import numpy as np

from .controlled import LearningConfig, MarketConfig, MarketEnv, QAgent, RuleAgent, integer



#### digest

In [16]:
%%inline_module agent02_assignment.code.training agent02_assignment/code/training.py both
def digest(value):
    encoded = json.dumps(value, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()
    return hashlib.sha256(encoded).hexdigest()



#### q_digest

In [17]:
%%inline_module agent02_assignment.code.training agent02_assignment/code/training.py both
def q_digest(q):
    array = np.asarray(q, dtype='<f8')
    if array.shape != (2, 2, 2) or not np.isfinite(array).all():
        raise ValueError('Expected finite 2x2x2 Q table')
    return hashlib.sha256(array.tobytes(order='C')).hexdigest()



#### TrainSession

In [18]:
%%inline_module agent02_assignment.code.training agent02_assignment/code/training.py both
class TrainSession:
    def __init__(self, market, learning, seed):
        self.market, self.learning, self.seed = market, learning, integer(seed, 'seed')
        self.env = MarketEnv(market, seed)
        # Distinct from demand and sales; shared across configs for paired seeds.
        self.agent = QAgent(learning, int(np.random.SeedSequence([seed, 3]).generate_state(1)[0]))
        self.rule = RuleAgent()
        self.completed, self.rows, self.trace = 0, [], []
        self.visits = np.zeros((2, 2, 2), dtype=np.int64)

    def run_chunk(self, episodes, *, trace_limit=0, guard=None):
        integer(episodes, 'chunk episodes', 1)
        integer(trace_limit, 'trace_limit')
        if self.completed + episodes > self.learning.episodes:
            raise ValueError('Chunk exceeds declared episode budget')
        for _ in range(episodes):
            if guard is not None:
                guard()
            states = self.env.reset()
            total, counts = np.zeros(2), np.zeros((2, 2), dtype=np.int64)
            epsilon = self.learning.epsilon_at(self.completed)
            for step in range(self.market.steps):
                # Neither policy sees the other's current action before choosing.
                actions = (self.agent.act(states[0], epsilon), self.rule.act(states[1]))
                result = self.env.step(actions)
                self.agent.learn(states[0], actions[0], result.rewards[0], result.states[0], result.done)
                self.visits[(*states[0], actions[0])] += 1
                total += result.rewards
                for role in range(2):
                    counts[role, actions[role]] += 1
                if len(self.trace) < trace_limit:
                    self.trace.append({'episode': self.completed+1, 'step': step+1,
                                       'states_before': [list(s) for s in states], 'actions': list(actions),
                                       'rewards': list(result.rewards), 'states_after': [list(s) for s in result.states],
                                       'sales': list(result.sales), 'terminal': result.done})
                states = result.states
            self.completed += 1
            self.rows.append({'episode': self.completed, 'rl_episode_reward': float(total[0]),
                              'rule_episode_reward': float(total[1]),
                              'rl_mean_reward_per_step': float(total[0] / self.market.steps),
                              'rule_mean_reward_per_step': float(total[1] / self.market.steps),
                              'epsilon': epsilon, 'alpha': self.learning.alpha, 'action_counts': counts.tolist()})

    def checkpoint(self):
        state = {'schema': 'ag02.controlled.checkpoint.1', 'market': asdict(self.market),
                 'learning': asdict(self.learning), 'seed': self.seed, 'completed': self.completed,
                 'q': self.agent.q.tolist(), 'visits': self.visits.tolist(), 'rows': self.rows, 'trace': self.trace,
                 'rng': {'demand': self.env.demand_rng.bit_generator.state,
                         'sales': [rng.bit_generator.state for rng in self.env.sales_rngs],
                         'agent': self.agent.rng.bit_generator.state}}
        return copy.deepcopy({**state, 'sha256': digest(state)})

    @classmethod
    def from_checkpoint(cls, saved):
        state = copy.deepcopy(saved)
        fingerprint = state.pop('sha256')
        if state['schema'] != 'ag02.controlled.checkpoint.1' or digest(state) != fingerprint:
            raise ValueError('Checkpoint schema/hash mismatch')
        result = cls(MarketConfig(**state['market']), LearningConfig(**state['learning']), state['seed'])
        result.completed = integer(state['completed'], 'completed')
        if result.completed > result.learning.episodes or len(state['rows']) != result.completed:
            raise ValueError('Checkpoint count mismatch')
        q_digest(state['q'])
        result.agent.q[:] = state['q']
        visits = np.asarray(state['visits'])
        if visits.shape != (2, 2, 2) or (visits < 0).any() or visits.sum() != result.completed * result.market.steps:
            raise ValueError('Checkpoint visits mismatch')
        result.visits[:] = visits
        result.rows, result.trace = state['rows'], state['trace']
        result.env.demand_rng.bit_generator.state = state['rng']['demand']
        for rng, value in zip(result.env.sales_rngs, state['rng']['sales'], strict=True):
            rng.bit_generator.state = value
        result.agent.rng.bit_generator.state = state['rng']['agent']
        return result



### `agent02_assignment.code.augmented`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [19]:
%%inline_module agent02_assignment.code.augmented agent02_assignment/code/augmented.py both
"""Tabular Q with own history and remaining time; market/reward stay unchanged."""

import copy

import hashlib

from dataclasses import asdict

import numpy as np

from .controlled import (
    LearningConfig,
    MarketConfig,
    MarketEnv,
    RuleAgent,
    binary,
    conversion,
    integer,
    margin,
)

from .training import digest

SHAPE = (2, 2, 2, 20, 2)



#### state_index

In [20]:
%%inline_module agent02_assignment.code.augmented agent02_assignment/code/augmented.py both
def state_index(state):
    if not isinstance(state, (tuple, list)) or len(state)!=4:
        raise ValueError('State must be demand,own_previous,other_previous,remaining')
    remaining = integer(state[3], 'remaining', 1)
    if remaining>20:
        raise ValueError('Remaining exceeds registered horizon20')
    return (*[binary(v) for v in state[:3]], remaining-1)



#### observe

In [21]:
%%inline_module agent02_assignment.code.augmented agent02_assignment/code/augmented.py both
def observe(env, role):
    binary(role, 'role')
    if not env.ready or env.elapsed>=env.config.steps:
        raise RuntimeError('Observation only exists for a live episode')
    return (env.demand, env.previous[role], env.previous[1-role], env.config.steps-env.elapsed)



#### augmented_hash

In [22]:
%%inline_module agent02_assignment.code.augmented agent02_assignment/code/augmented.py both
def augmented_hash(q):
    array = np.asarray(q, dtype='<f8')
    if array.shape!=SHAPE or not np.isfinite(array).all():
        raise ValueError('Expected finite2x2x2x20x2 augmented Q')
    return hashlib.sha256(array.tobytes(order='C')).hexdigest()



#### alias_witness

In [23]:
%%inline_module agent02_assignment.code.augmented agent02_assignment/code/augmented.py both
def alias_witness():
    market, rule = MarketConfig(), RuleAgent()
    observations = [(1, own, 0, 20) for own in (0, 1)]
    other = [rule.act((1, own)) for own in (0, 1)]
    return {'old_observation_a': [1, 0], 'old_observation_b': [1, 0],
        'new_observation_a': list(observations[0]), 'new_observation_b': list(observations[1]),
        'rule_current_actions': other,
        'expected_reward_keep_price': [market.customers_high*conversion(1, 0, a)*margin(market, 0) for a in other],
        'interpretation': 'Same old local observation hides own previous action that changes fixed rule current action; analytic fixture, not a final trajectory'}



#### AugmentedAgent

In [24]:
%%inline_module agent02_assignment.code.augmented agent02_assignment/code/augmented.py both
class AugmentedAgent:
    def __init__(self, learning, seed):
        self.config = learning
        self.rng = np.random.default_rng(integer(seed, 'seed'))
        self.q = np.zeros(SHAPE, dtype=np.float64)

    def act(self, state, epsilon, *, explore=True):
        values = self.q[state_index(state)]
        if not np.isfinite(epsilon) or not 0<=epsilon<=1 or not np.isfinite(values).all():
            raise ValueError('Invalid epsilon or Q')
        if explore and self.rng.random()<epsilon:
            return int(self.rng.integers(2))
        best = np.flatnonzero(np.isclose(values, values.max(), rtol=1e-5, atol=1e-8))
        return int(self.rng.choice(best))

    def learn(self, state, action, reward, following, terminal):
        index = state_index(state)
        action = binary(action, 'action')
        if not np.isfinite(reward) or bool(terminal)!=(state[3]==1):
            raise ValueError('Invalid reward/terminal time')
        if terminal:
            target = reward
        else:
            following_index = state_index(following)
            if following[3]!=state[3]-1:
                raise ValueError('Remaining time must decrease exactly1')
            target = reward+self.config.gamma*float(self.q[following_index].max())
        key = (*index, action)
        self.q[key] += self.config.alpha*(target-self.q[key])
        if not np.isfinite(self.q[key]):
            raise ValueError('Nonfinite augmented Q update')



#### AugmentedSession

In [25]:
%%inline_module agent02_assignment.code.augmented agent02_assignment/code/augmented.py both
class AugmentedSession:
    def __init__(self, market, learning, seed):
        if market.steps>20:
            raise ValueError('Augmented table supports horizon<=20')
        self.market, self.learning, self.seed = market, learning, integer(seed, 'seed')
        self.env, self.rule = MarketEnv(market, seed), RuleAgent()
        self.agent = AugmentedAgent(learning, int(np.random.SeedSequence([seed, 3]).generate_state(1)[0]))
        self.completed, self.rows, self.trace = 0, [], []
        self.visits = np.zeros(SHAPE, dtype=np.int64)

    def run_chunk(self, episodes, *, trace_limit=0, guard=None):
        integer(episodes, 'episodes', 1)
        integer(trace_limit, 'trace_limit')
        if self.completed+episodes>self.learning.episodes:
            raise ValueError('Chunk exceeds registered episode budget')
        for _ in range(episodes):
            if guard is not None:
                guard()
            states = self.env.reset()
            total, counts = np.zeros(2), np.zeros((2, 2), dtype=np.int64)
            epsilon = self.learning.epsilon_at(self.completed)
            for step in range(self.market.steps):
                observation = observe(self.env, 0)
                actions = (self.agent.act(observation, epsilon), self.rule.act(states[1]))
                result = self.env.step(actions)
                following = None if result.done else observe(self.env, 0)
                self.agent.learn(observation, actions[0], result.rewards[0], following, result.done)
                self.visits[(*state_index(observation), actions[0])] += 1
                total += result.rewards
                for role in (0, 1):
                    counts[role, actions[role]] += 1
                if len(self.trace)<trace_limit:
                    self.trace.append({'episode': self.completed+1, 'step': step+1,
                        'states_before': [list(s) for s in states], 'augmented_state_before': list(observation),
                        'actions': list(actions), 'rewards': list(result.rewards),
                        'augmented_state_after': list(following) if following is not None else None,
                        'states_after': [list(s) for s in result.states], 'sales': list(result.sales), 'terminal': result.done})
                states = result.states
            self.completed += 1
            self.rows.append({'episode': self.completed, 'rl_episode_reward': float(total[0]),
                'rule_episode_reward': float(total[1]), 'rl_mean_reward_per_step': float(total[0]/self.market.steps),
                'rule_mean_reward_per_step': float(total[1]/self.market.steps), 'epsilon': epsilon,
                'alpha': self.learning.alpha, 'action_counts': counts.tolist()})

    def checkpoint(self):
        value = {'schema': 'ag02.augmented.checkpoint.1', 'market': asdict(self.market), 'learning': asdict(self.learning),
            'seed': self.seed, 'completed': self.completed, 'q': self.agent.q.tolist(), 'visits': self.visits.tolist(),
            'rows': self.rows, 'trace': self.trace,
            'rng': {'demand': self.env.demand_rng.bit_generator.state,
                'sales': [r.bit_generator.state for r in self.env.sales_rngs], 'agent': self.agent.rng.bit_generator.state}}
        return copy.deepcopy({**value, 'sha256': digest(value)})

    @classmethod
    def from_checkpoint(cls, saved):
        value = copy.deepcopy(saved)
        fingerprint = value.pop('sha256')
        if value['schema']!='ag02.augmented.checkpoint.1' or digest(value)!=fingerprint:
            raise ValueError('Augmented checkpoint schema/hash mismatch')
        result = cls(MarketConfig(**value['market']), LearningConfig(**value['learning']), value['seed'])
        result.completed = integer(value['completed'], 'completed')
        if result.completed>result.learning.episodes or len(value['rows'])!=result.completed:
            raise ValueError('Checkpoint episode count mismatch')
        augmented_hash(value['q'])
        result.agent.q[:] = value['q']
        visits = np.asarray(value['visits'])
        if visits.shape!=SHAPE or (visits<0).any() or visits.sum()!=result.completed*result.market.steps:
            raise ValueError('Checkpoint visits mismatch')
        result.visits[:] = visits
        result.rows, result.trace = value['rows'], value['trace']
        result.env.demand_rng.bit_generator.state = value['rng']['demand']
        for rng, state in zip(result.env.sales_rngs, value['rng']['sales'], strict=True):
            rng.bit_generator.state = state
        result.agent.rng.bit_generator.state = value['rng']['agent']
        return result



### `agent02_assignment.workspace`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [26]:
%%inline_module agent02_assignment.workspace agent02_assignment/workspace.py both
"""Small, tested workspace guards; never import or run the teacher's program."""

import ast

import hashlib

import json

from pathlib import Path, PureWindowsPath



#### inside

In [27]:
%%inline_module agent02_assignment.workspace agent02_assignment/workspace.py both
def inside(base: Path, relative: str) -> Path:
    """Resolve a portable relative target and reject traversal and symlink escape."""
    windows = PureWindowsPath(relative)
    if not relative or windows.drive or windows.root or '\\' in relative:
        raise ValueError('Target must be a nonempty portable relative path')
    target = (base / relative).resolve()
    if Path(relative).is_absolute() or not target.is_relative_to(base.resolve()):
        raise ValueError('Target escapes the assignment workspace')
    return target



#### sha256_file

In [28]:
%%inline_module agent02_assignment.workspace agent02_assignment/workspace.py both
def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()



#### copy_verified

In [29]:
%%inline_module agent02_assignment.workspace agent02_assignment/workspace.py both
def copy_verified(source: Path, base: Path, relative: str, expected: str) -> dict:
    """Copy exact bytes once; reuse identical files and refuse user-file overwrite."""
    target = inside(base, relative)
    content = source.read_bytes()
    actual = hashlib.sha256(content).hexdigest()
    if actual != expected:
        raise ValueError(f'Source hash mismatch: {source.name}')
    operation = 'reused'
    if target.exists():
        if sha256_file(target) != expected:
            raise FileExistsError(f'Refusing to overwrite different target: {target}')
    else:
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open('xb') as stream:
            stream.write(content)
        operation = 'copied'
    return {'path': relative, 'bytes': len(content), 'sha256': actual, 'operation': operation}



#### inspect_python

In [30]:
%%inline_module agent02_assignment.workspace agent02_assignment/workspace.py both
def inspect_python(path: Path) -> dict:
    """Parse and compile without exec/import, preserving original newline bytes."""
    content = path.read_bytes()
    tree = ast.parse(content, filename=str(path))
    compile(tree, str(path), 'exec')
    classes = [node.name for node in tree.body if isinstance(node, ast.ClassDef)]
    functions = [node.name for node in tree.body if isinstance(node, ast.FunctionDef)]
    defaults = {}
    for node in tree.body:
        if isinstance(node, ast.ClassDef) and node.name == 'Config':
            for field in node.body:
                if isinstance(field, ast.AnnAssign) and isinstance(field.target, ast.Name):
                    defaults[field.target.id] = ast.literal_eval(field.value)
    return {'sha256': hashlib.sha256(content).hexdigest(), 'bytes': len(content),
            'lines': len(content.splitlines()), 'syntax_valid': True,
            'execution_performed': False, 'classes': classes, 'functions': functions,
            'config_defaults': defaults}



#### prepare_sources

In [31]:
%%inline_module agent02_assignment.workspace agent02_assignment/workspace.py both
def prepare_sources(source_root: Path, base: Path, manifest: list) -> dict:
    """Validate the whole manifest before writing any original-source copy."""
    if not manifest:
        raise ValueError('Source manifest is empty')
    for record in manifest:
        name = record['name']
        if Path(name).name != name or PureWindowsPath(name).name != name or name in {'.', '..'}:
            raise ValueError('Source name must be a basename')
        source = inside(source_root, name)
        if source.stat().st_size != record['bytes']:
            raise ValueError(f'Source size mismatch: {name}')
        if sha256_file(source) != record['sha256']:
            raise ValueError(f'Source hash mismatch: {name}')
    copies = [copy_verified(source_root / record['name'], base,
                            'sources/provided/' + record['name'], record['sha256'])
              for record in manifest]
    code_record = next(record for record in manifest if record['name'] == 'agent02_code.tex')
    reference = copy_verified(source_root / code_record['name'], base,
                              'code/agent02_experiment.py', code_record['sha256'])
    inspection = inspect_python(base / reference['path'])
    return {'sources': copies, 'reference': reference, 'inspection': inspection}



#### validate_runtime

In [32]:
%%inline_module agent02_assignment.workspace agent02_assignment/workspace.py both
def validate_runtime(actual: dict, expected: dict) -> None:
    if actual['python'] != expected['python']:
        raise ValueError('Python version changed from the pinned runtime')
    for name, version in expected.get('packages', {}).items():
        if actual['packages'].get(name) != version:
            raise ValueError(f'Pinned package mismatch: {name}')



#### write_receipt

In [33]:
%%inline_module agent02_assignment.workspace agent02_assignment/workspace.py both
def write_receipt(base: Path, relative: str, value: dict) -> Path:
    """New generated JSON artifact, with explicit no-overwrite semantics."""
    target = inside(base, relative)
    if target.exists():
        if json.loads(target.read_text(encoding='utf-8')) != value:
            raise FileExistsError(f'Receipt already exists with different content: {target}')
        return target
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open('x', encoding='utf-8', newline='\n') as stream:
        stream.write(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True) + '\n')
    return target



### `agent02_assignment.code.reference_adapter`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [34]:
%%inline_module agent02_assignment.code.reference_adapter agent02_assignment/code/reference_adapter.py both
"""Guarded import of the exact local teacher snapshot; never call its main driver."""

import importlib.util

import json

import sys

import time

from pathlib import Path

from ..workspace import sha256_file

from .controlled import integer

BASE = Path(__file__).resolve().parents[1]



#### load_reference

In [35]:
%%inline_module agent02_assignment.code.reference_adapter agent02_assignment/code/reference_adapter.py both
def load_reference(path=None):
    expected_path = BASE / 'code/agent02_experiment.py'
    source = expected_path if path is None else Path(path)
    lock = json.loads((BASE / 'evidence/phase01/input_lock.json').read_text())
    expected_hash = next(row['sha256'] for row in lock['files'] if row['path'] == 'code/agent02_experiment.py')
    if source.resolve() != expected_path.resolve() or sha256_file(source) != expected_hash:
        raise ValueError('Only the unchanged Phase 1 reference snapshot may be imported')
    module_name = '_ag02_teacher_reference_v1'
    spec = importlib.util.spec_from_file_location(module_name, source)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module



#### reference_smoke

In [36]:
%%inline_module agent02_assignment.code.reference_adapter agent02_assignment/code/reference_adapter.py both
def reference_smoke(*, episodes=100, seed=730042):
    integer(episodes, 'episodes', 1)
    integer(seed, 'seed')
    if episodes > 100:
        raise ValueError('Reference smoke is limited to 100 episodes; full runs belong to Phase 3')
    reference = load_reference()
    config = reference.Config(episodes=episodes)
    started = time.perf_counter()
    learner, rows = reference.train_rl_against_rule(config, seed=seed)
    return {'purpose': 'smoke_only_not_experimental_result', 'seed': seed,
            'episodes': episodes, 'market_steps': episodes * config.steps_per_episode,
            'wall_seconds': time.perf_counter() - started,
            'q': learner.q.tolist(), 'rows': rows,
            'reference_sha256': sha256_file(BASE / 'code/agent02_experiment.py'),
            'main_called': False, 'evaluation_performed': False}



### `agent02_assignment.code.reference_training`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [37]:
%%inline_module agent02_assignment.code.reference_training agent02_assignment/code/reference_training.py both
"""Episode-boundary instrumentation around unchanged teacher classes and RNG order."""

import copy

import numpy as np

from .controlled import LearningConfig, MarketConfig, integer

from .reference_adapter import load_reference

from .training import digest, q_digest



#### ReferenceSession

In [38]:
%%inline_module agent02_assignment.code.reference_training agent02_assignment/code/reference_training.py both
class ReferenceSession:
    """Same loop/calls as train_rl_against_rule; added logging makes no RNG draws."""

    def __init__(self, market, learning, seed):
        if market != MarketConfig():
            raise ValueError('REF adapter only supports the unchanged teaching market')
        self.market, self.learning, self.seed = market, learning, integer(seed, 'seed')
        teacher = load_reference()
        self.config = teacher.Config(episodes=learning.episodes, alpha=learning.alpha, gamma=learning.gamma,
                                     epsilon_start=learning.epsilon_start, epsilon_min=learning.epsilon_min,
                                     epsilon_decay=learning.epsilon_decay)
        self.env = teacher.ECommerceEnvironment(self.config, seed=seed)
        self.agent = teacher.QLearningAgent(alpha=learning.alpha, gamma=learning.gamma,
                                            epsilon=learning.epsilon_start, seed=seed+10000)
        self.rule = teacher.RuleBasedAgent()
        self.completed, self.rows, self.trace = 0, [], []
        self.visits = np.zeros((2, 2, 2), dtype=np.int64)

    def run_chunk(self, episodes, *, trace_limit=0, guard=None):
        integer(episodes, 'chunk episodes', 1)
        integer(trace_limit, 'trace_limit')
        if self.completed + episodes > self.learning.episodes:
            raise ValueError('Chunk exceeds declared episode budget')
        for _ in range(episodes):
            if guard is not None:
                guard()
            self.agent.epsilon = self.learning.epsilon_at(self.completed)
            states = self.env.reset()
            totals = [0., 0.]
            counts = np.zeros((2, 2), dtype=np.int64)
            for step in range(self.market.steps):
                actions = [self.agent.act(states[0], explore=True), self.rule.act(states[1])]
                next_states, rewards, info = self.env.step(actions)
                terminal = step == self.market.steps - 1
                self.agent.learn(state=states[0], action=actions[0], reward=rewards[0],
                                 next_state=next_states[0], terminal=terminal)
                self.visits[(*states[0], actions[0])] += 1
                for role in (0, 1):
                    totals[role] += rewards[role]
                    counts[role, actions[role]] += 1
                if len(self.trace) < trace_limit:
                    self.trace.append({'episode': self.completed+1, 'step': step+1,
                                       'states_before': [list(s) for s in states], 'actions': actions,
                                       'rewards': rewards, 'states_after': [list(s) for s in next_states],
                                       'sales': info['sales'], 'terminal': terminal})
                states = next_states
            self.completed += 1
            self.rows.append({'episode': self.completed, 'rl_episode_reward': totals[0],
                              'rule_episode_reward': totals[1],
                              'rl_mean_reward_per_step': totals[0]/self.market.steps,
                              'rule_mean_reward_per_step': totals[1]/self.market.steps,
                              'epsilon': self.agent.epsilon, 'alpha': self.learning.alpha,
                              'action_counts': counts.tolist()})

    def checkpoint(self):
        from dataclasses import asdict

        state = {'schema': 'ag02.reference.checkpoint.1', 'market': asdict(self.market),
                 'learning': asdict(self.learning), 'seed': self.seed, 'completed': self.completed,
                 'q': self.agent.q.tolist(), 'rows': self.rows, 'trace': self.trace,
                 'visits': self.visits.tolist(), 'epsilon': self.agent.epsilon,
                 'rng': {'environment': self.env.rng.bit_generator.state,
                         'agent': self.agent.rng.bit_generator.state}}
        return copy.deepcopy({**state, 'sha256': digest(state)})

    @classmethod
    def from_checkpoint(cls, saved):
        state = copy.deepcopy(saved)
        fingerprint = state.pop('sha256')
        if state['schema'] != 'ag02.reference.checkpoint.1' or digest(state) != fingerprint:
            raise ValueError('REF checkpoint schema/hash mismatch')
        result = cls(MarketConfig(**state['market']), LearningConfig(**state['learning']), state['seed'])
        result.completed = integer(state['completed'], 'completed')
        if result.completed > result.learning.episodes or len(state['rows']) != result.completed:
            raise ValueError('REF checkpoint count mismatch')
        q_digest(state['q'])
        result.agent.q[:] = state['q']
        visits = np.asarray(state['visits'])
        if visits.shape != (2, 2, 2) or (visits < 0).any() or visits.sum() != result.completed * result.market.steps:
            raise ValueError('REF checkpoint visitation mismatch')
        result.visits[:] = visits
        result.rows, result.trace = state['rows'], state['trace']
        result.agent.epsilon = state['epsilon']
        result.env.rng.bit_generator.state = state['rng']['environment']
        result.agent.rng.bit_generator.state = state['rng']['agent']
        return result



### `agent02_assignment.code.evaluation.frozen`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [39]:
%%inline_module agent02_assignment.code.evaluation.frozen agent02_assignment/code/evaluation/frozen.py both
"""Matched exogenous scenarios, frozen Q copy, and separate evaluation tie RNG."""

import numpy as np

from ..controlled import LearningConfig, MarketEnv, QAgent, RuleAgent, Tape, binary, integer

from ..training import q_digest



#### evaluate_dev

In [40]:
%%inline_module agent02_assignment.code.evaluation.frozen agent02_assignment/code/evaluation/frozen.py both
def evaluate_dev(q, market, *, episodes, scenario_seed, role=0, partition='development'):
    if partition not in {'fixture', 'development'}:
        raise PermissionError('Final evaluation requires the later declared evaluator/gate')
    integer(episodes, 'episodes', 1)
    binary(role, 'role')
    integer(scenario_seed, 'scenario_seed')
    if episodes > 100:
        raise ValueError('Phase 2 DEV evaluation is limited to 100 episodes')
    before = q_digest(q)
    learner = QAgent(LearningConfig(), int(np.random.SeedSequence([scenario_seed, 4]).generate_state(1)[0]))
    learner.q[:] = q
    learner.q.flags.writeable = False
    rule = RuleAgent()
    env = MarketEnv(market, scenario_seed)
    totals, action_counts, rows = np.zeros(2), np.zeros((2, 2), dtype=np.int64), []
    for episode in range(episodes):
        states = env.reset(Tape.generate(market, scenario_seed, episode))
        reward = np.zeros(2)
        for _ in range(market.steps):
            actions = [0, 0]
            actions[role] = learner.act(states[role], 0, explore=False)
            actions[1-role] = rule.act(states[1-role])
            result = env.step(actions)
            reward += result.rewards
            for index in range(2):
                action_counts[index, actions[index]] += 1
            states = result.states
        totals += reward
        rows.append({'episode': episode+1, 'learner_reward': float(reward[role]),
                     'rule_reward': float(reward[1-role])})
    after = q_digest(q)
    if before != after or q_digest(learner.q) != before:
        raise RuntimeError('Frozen Q changed during evaluation')
    return {'partition': partition, 'role': role, 'scenario_seed': scenario_seed,
            'episodes': episodes, 'steps': episodes * market.steps, 'rows': rows,
            'total_reward': float(totals[role]), 'mean_reward': float(totals[role] / (episodes * market.steps)),
            'rule_total_reward': float(totals[1-role]),
            'learner_action_counts': action_counts[role].tolist(), 'q_hash_before': before,
            'q_hash_after': after, 'exploration': False, 'tie_break': 'seeded_np_isclose_1e-5_1e-8'}



### `agent02_assignment.phase01`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [41]:
%%inline_module agent02_assignment.phase01 agent02_assignment/phase01.py both
"""Phase 1 preparation and read-only preflight; no training command is exposed."""

import argparse

import importlib.metadata

import json

import os

import platform

import shutil

import sys

import tempfile

from pathlib import Path

from .workspace import inside, inspect_python, prepare_sources, sha256_file, validate_runtime, write_receipt

BASE = Path(__file__).resolve().parent

REPO = BASE.parent

SOURCE = Path('/mnt/c/Users/anhca/Documents/Intel_sys_dev/ASIGNMENT-AGENT_02')

EXPECTED_PYTHON = '/home/tuananh/.local/share/ai-term-paper-v3/.venv-torch/bin/python'

FOLDERS = ('docs', 'code/agents', 'code/environments', 'code/evaluation', 'configs',
           'notebooks', 'data/derived', 'artifacts/runs', 'artifacts/metrics',
           'artifacts/figures', 'artifacts/manifests', 'report/sections', 'report/build',
           'report/qa', 'output', 'scripts')



#### runtime_inventory

In [42]:
%%inline_module agent02_assignment.phase01 agent02_assignment/phase01.py both
def runtime_inventory():
    expected = json.loads((BASE / 'evidence/planning_runtime.json').read_text())
    versions = {name: importlib.metadata.version(name) for name in expected['packages']}
    actual = {'python': platform.python_version(), 'packages': versions,
              'executable': sys.executable, 'user': os.environ.get('USER'),
              'platform': platform.system()}
    validate_runtime(actual, expected)
    if sys.executable != EXPECTED_PYTHON or actual['user'] != 'tuananh' or actual['platform'] != 'Linux':
        raise ValueError('Use the existing Ubuntu/tuananh pinned runtime via run_experiment.ps1')
    return actual



#### preserved_inputs

In [43]:
%%inline_module agent02_assignment.phase01 agent02_assignment/phase01.py both
def preserved_inputs():
    inventory = json.loads((BASE / 'evidence/planning_runtime.json').read_text())
    checks = []
    for record in inventory['assets']:
        actual = sha256_file(inside(REPO, record['path']))
        if actual != record['sha256']:
            raise ValueError('Existing project input changed: ' + record['path'])
        checks.append({'path': record['path'], 'sha256': actual, 'unchanged': True})
    return checks



#### notebook_probe

In [44]:
%%inline_module agent02_assignment.phase01 agent02_assignment/phase01.py both
def notebook_probe():
    import matplotlib
    import nbformat
    import numpy as np
    from jupyter_client import KernelManager
    from nbclient import NotebookClient

    notebook = nbformat.v4.new_notebook(cells=[
        nbformat.v4.new_markdown_cell(
            '# Agent02 Phase 1 runtime probe\n\nEnvironment check only. No teacher code, '
            'Q-learning, training, data fitting or final evaluation is executed.'),
        nbformat.v4.new_code_cell(
            'import platform, sys\n'
            'print("Python:", platform.python_version())\nprint("Executable:", sys.executable)'),
        nbformat.v4.new_code_cell(
            'import numpy as np\nimport matplotlib\nmatplotlib.use("Agg")\n'
            'import matplotlib.pyplot as plt\n'
            'print("NumPy:", np.__version__, "| Matplotlib:", matplotlib.__version__)\n'
            'print("Array sum:", int(np.arange(4).sum()))\n'
            'print("Charts generated: 0 | Training steps: 0")'),
    ])
    notebook.metadata.kernelspec = {'display_name': 'Agent02 pinned WSL Python',
                                     'language': 'python', 'name': 'python3'}
    # DrvFS cannot enforce mode 0600. A private Linux temp directory keeps Jupyter's
    # secure_write protection enabled; never set JUPYTER_ALLOW_INSECURE_WRITES.
    with tempfile.TemporaryDirectory(prefix='ag02-phase01-') as temporary:
        cache = Path(temporary)
        manager = KernelManager(kernel_name='python3', connection_file=str(cache / 'connection.json'))
        manager.kernel_spec.argv = [sys.executable, '-m', 'ipykernel_launcher', '-f', '{connection_file}']
        client = NotebookClient(notebook, km=manager, timeout=60, allow_errors=False,
                                resources={'metadata': {'path': str(BASE)}})
        try:
            client.execute(env={**os.environ, 'PYTHONDONTWRITEBYTECODE': '1', 'CUDA_VISIBLE_DEVICES': '',
                                'MPLBACKEND': 'Agg', 'IPYTHONDIR': str(cache / 'ipython'),
                                'JUPYTER_RUNTIME_DIR': str(cache / 'jupyter'),
                                'MPLCONFIGDIR': str(cache / 'matplotlib')})
        finally:
            if manager.has_kernel:
                manager.shutdown_kernel(now=True)
    target = inside(BASE, 'evidence/phase01/runtime_probe.ipynb')
    if target.exists():
        raise FileExistsError('Keep previous notebook probe; use a new phase attempt')
    nbformat.write(notebook, target)
    outputs = [''.join(output.get('text', '') for output in cell.outputs
                        if output.output_type == 'stream')
               for cell in notebook.cells if cell.cell_type == 'code']
    if len(outputs) != 2 or EXPECTED_PYTHON not in outputs[0] or 'Array sum: 6' not in outputs[1]:
        raise ValueError('Runtime notebook did not use the expected interpreter/outputs')
    return {'path': str(target.relative_to(BASE)), 'sha256': sha256_file(target),
            'executed_code_cells': 2, 'outputs': outputs,
            'numpy': np.__version__, 'matplotlib': matplotlib.__version__,
            'charts_generated': 0, 'training_steps': 0}



#### prepare

In [45]:
%%inline_module agent02_assignment.phase01 agent02_assignment/phase01.py both
def prepare():
    if (BASE / 'evidence/phase01/preparation.json').exists():
        raise FileExistsError('Phase 1 already prepared; use read-only preflight')
    free_bytes = shutil.disk_usage(BASE).free
    if free_bytes < 5 * 1024 ** 3:
        raise RuntimeError('Host workspace volume has less than 5 GiB free; do not install or delete data')
    runtime = runtime_inventory()
    before = preserved_inputs()
    manifest = json.loads((BASE / 'sources/manifest.json').read_text())
    source_receipt = prepare_sources(SOURCE, BASE, manifest)
    for relative in FOLDERS:
        inside(BASE, relative).mkdir(parents=True, exist_ok=True)
    existing_receipt = BASE / 'evidence/phase01/source_receipt.json'
    if existing_receipt.exists():
        original = json.loads(existing_receipt.read_text())
        original_pairs = [(row['path'], row['sha256']) for row in original['sources']]
        current_pairs = [(row['path'], row['sha256']) for row in source_receipt['sources']]
        if original['inspection'] != source_receipt['inspection'] or original_pairs != current_pairs:
            raise ValueError('Earlier source receipt differs; retain evidence and investigate')
    else:
        write_receipt(BASE, 'evidence/phase01/source_receipt.json', source_receipt)
    write_receipt(BASE, 'evidence/phase01/reference_inspection.json', source_receipt['inspection'])
    probe = notebook_probe()
    receipt = {'status': 'PASS', 'phase': 1, 'runtime': runtime, 'host_free_bytes_before': free_bytes,
               'notebook_probe': probe,
               'preservation_before': before, 'preservation_after': preserved_inputs(),
               'folders_created': list(FOLDERS), 'reference_executed': False,
               'training_started': False, 'final_evaluation_started': False}
    write_receipt(BASE, 'evidence/phase01/preparation.json', receipt)
    return receipt



#### preflight

In [46]:
%%inline_module agent02_assignment.phase01 agent02_assignment/phase01.py both
def preflight():
    runtime = runtime_inventory()
    manifest = json.loads((BASE / 'sources/manifest.json').read_text())
    for record in manifest:
        for folder in (SOURCE, BASE / 'sources/provided'):
            target = inside(folder, record['name'])
            if target.stat().st_size != record['bytes'] or sha256_file(target) != record['sha256']:
                raise ValueError('Source changed: ' + str(target))
    reference = inspect_python(BASE / 'code/agent02_experiment.py')
    expected = next(record['sha256'] for record in manifest if record['name'] == 'agent02_code.tex')
    if reference['sha256'] != expected:
        raise ValueError('Reference Python copy changed')
    return {'status': 'PASS', 'runtime': runtime, 'source_hashes_verified': len(manifest),
            'reference': reference, 'preservation': preserved_inputs(), 'read_only': True,
            'training_started': False, 'final_evaluation_started': False}



#### main

In [47]:
%%inline_module agent02_assignment.phase01 agent02_assignment/phase01.py both
def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('task', choices=('prepare', 'preflight', 'inspect-reference'))
    args = parser.parse_args()
    if args.task == 'prepare':
        receipt = prepare()
    elif args.task == 'preflight':
        receipt = preflight()
    else:
        receipt = inspect_python(BASE / 'code/agent02_experiment.py')
    print(json.dumps(receipt, ensure_ascii=False, indent=2))



#### Thiết lập / điểm vào

In [48]:
%%inline_module agent02_assignment.phase01 agent02_assignment/phase01.py both
if __name__ == '__main__':
    main()



### `agent02_assignment.protocol`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [49]:
%%inline_module agent02_assignment.protocol agent02_assignment/protocol.py both
"""A priori protocol builder; no model execution, raw-price parsing or TEST access."""

import json

from dataclasses import asdict, replace

from pathlib import Path

from .code.controlled import LearningConfig, MarketConfig

from .workspace import sha256_file

BASE = Path(__file__).resolve().parent



#### build_protocol

In [50]:
%%inline_module agent02_assignment.protocol agent02_assignment/protocol.py both
def build_protocol():
    seeds = [11, 22, 33, 44, 55]
    default = LearningConfig()
    configs = {'baseline': asdict(default), 'alpha_low': asdict(replace(default, alpha=.01)),
               'alpha_high': asdict(replace(default, alpha=.50)),
               'epsilon_low_fixed': asdict(replace(default, epsilon_start=.05, epsilon_min=.05, epsilon_decay=1)),
               'epsilon_high_fixed': asdict(replace(default, epsilon_start=.30, epsilon_min=.30, epsilon_decay=1))}
    runs = [{'run_id': 'ref/baseline/s0042', 'track': 'REF', 'phase': 3, 'seed': 42,
             'config_id': 'baseline', 'kernel_id': 'iid', 'learners': 1, 'status': 'declared_not_started'}]
    for config_id in configs:
        for seed in seeds:
            runs.append({'run_id': f'controlled/iid/{config_id}/s{seed:04d}', 'track': 'CONTROLLED',
                         'phase': 3 if config_id == 'baseline' and seed == 11 else 4,
                         'seed': seed, 'config_id': config_id, 'kernel_id': 'iid', 'learners': 1,
                         'status': 'declared_not_started'})
    groups = [('two_learners', ('iid',), 6, 2), ('retrain', ('persistent', 'gold'), 6, 1),
              ('augmented', ('iid', 'persistent', 'gold'), 7, 1)]
    for group, kernels, phase, learners in groups:
        for kernel in kernels:
            for seed in seeds:
                runs.append({'run_id': f'controlled/{kernel}/{group}/s{seed:04d}', 'track': 'CONTROLLED',
                             'phase': phase, 'seed': seed, 'config_id': group, 'kernel_id': kernel,
                             'learners': learners, 'status': 'declared_not_started'})
    split_path = BASE.parent / 'data/splits/v3/ch4_gold_pink_sheet/split_contract.json'
    split = json.loads(split_path.read_text())
    cutoff = next(item['target_end'][:7] for item in split['partitions'] if item['name'] == 'train')
    return {
        'schema_version': 'ag02.protocol.1', 'local_date': '2026-10-24', 'device': 'CPU NumPy float64',
        'scope': 'Agent02 only; no LLM/MAS project, web or V3 TEST replay',
        'market': asdict(MarketConfig()), 'controlled_configs': configs, 'train_seeds': seeds,
        'training_runs': runs,
        'budget': {'train_episodes': 5000, 'steps_per_episode': 20,
                   'final_evaluation_episodes_per_role': 500, 'declared_market_runs': 56,
                   'declared_train_market_steps': 5600000, 'declared_learner_instances': 61,
                   'minimum_free_disk_bytes': 5 * 1024**3, 'trace_rows_per_run': 20,
                   'checkpoint_interval_episodes': 100, 'maximum_parallel_train_writers': 1},
        'rng': {'controlled_training': 'PCG64 SeedSequence([train_seed, channel]); channels demand=0,sales=1,2,policy=3',
                'policy_seed_derivation': 'int(SeedSequence([seed,3]).generate_state(1)[0])',
                'reference': 'Original single env RNG seed; action RNG seed+10000; reference code unchanged',
                'tapes': 'PCG64 SeedSequence([scenario_seed,episode,channel]); demand=0,customer_uniforms=1',
                'evaluation_ties': 'PCG64 int(SeedSequence([scenario_seed,4]).generate_state(1)[0])',
                'near_ties': {'rule': 'random among np.isclose to max', 'rtol': 1e-5, 'atol': 1e-8},
                'tape_sampling': 'Count independent uniforms < conversion for N customers; Binomial marginal'},
        'evaluation': {'roles': [0, 1], 'development_scenario_seeds': [1000000+s for s in seeds],
                       'final_scenario_seeds': [2000000+s for s in seeds],
                       'reference_native_final_seed': 500042, 'reference_native_final_phase': 5,
                       'final_evaluation_authorized_in_phase2': False, 'final_gate_phase': 5,
                       'future_variant_gate_phase': 6, 'future_improved_gate_phase': 7,
                       'fixed_opponent': 'Original rule; rule-vs-rule and no-discount diagnostics',
                       'paired_scenarios': 'Same scenario seed and episode tape across configs; both physical roles',
                       'frozen': 'Copy Q read-only, no learn, explore=False; verify source Q hash and RNG unchanged',
                       'aggregation': 'mean and sample std(ddof=1) across 5 training seeds after averaging roles',
                       'uncertainty': 'Report seed-level paired deltas; if CI used, resample seed pairs, not steps',
                       'selection': 'Report all preregistered configs; never tune/select on final results',
                       'journal_key': '(study_id,policy_sha256,scenario_spec_sha256,role); reuse identical saved output',
                       'recovery': 'Retain failed/partial claims; investigate/version, do not silently replay final'},
        'future_variants': {
            'persistent': {'kernel': [[.9, .1], [.1, .9]], 'initial_demand': 'Bernoulli(0.5)'},
            'gold': {'calibration_status': 'deferred_to_phase6', 'last_allowed_month': cutoff,
                     'source_path': 'data/raw/ch4/gold/CMO-Historical-Data-Monthly.xlsx',
                     'source_sha256': 'ea1c350827878ea3bbe30e3cda16a13fd3bd5b409b8458940dc94a36b5a33154',
                     'parent_split_path': str(split_path.relative_to(BASE.parent)),
                     'parent_split_sha256': sha256_file(split_path),
                     'regime': '1 if gold[t] > gold[t-1] else 0; consecutive observed calendar months only',
                     'transition_estimator': '(count_ij+1)/(sum_j count_ij+2)',
                     'smoothing_pseudocount': 1, 'initial_demand': 'Bernoulli(0.5)',
                     'missing_policy': 'Do not impute or bridge a missing month; record exclusions',
                     'real_demand_measurement': False, 'interpretation': 'synthetic demand proxy, not causality or investment'}},
        'improved_design': {'implementation_phase': 7, 'state': ['demand', 'own_previous', 'competitor_previous',
                                                              'remaining_steps'],
                            'q_entries': 320, 'remaining_steps': '1..20, zero future value at terminal',
                            'learning_parameters': asdict(default), 'reward': 'unchanged profit',
                            'oracle': 'optional exact finite-horizon known-model/fixed-rule check, never train labels'},
        'smoke': {'episodes_per_track': 100, 'steps_per_episode': 20, 'reference_seed': 730042,
                  'controlled_seed': 730011, 'development_scenario_seed': 1730011,
                  'development_episodes_per_role': 3, 'use_as_report_results': False},
        'outputs_per_run': ['manifest.json', 'episode_rewards.csv', 'checkpoint_eNNNNNN.json',
                            'q_table.json', 'policy.json', 'visits.json', 'trace_sample.json'],
        'figure_policy': {'format': 'black/grayscale', 'moving_average_episodes': 100,
                          'interpretation_notes_required': True, 'notes_template': 'docs/figure_explanation_template.md',
                          'required_notes': ['purpose', 'data_source', 'axes_and_units', 'legend', 'aggregation_and_n',
                                             'observations', 'quantitative_comparison', 'limitations']},
        'test_plan': ['8 state-action pairs and 4 simultaneous joint actions', 'all rule branches and reward margins',
                      'Q=4.65 calculation, terminal update, epsilon/ties', 'independent demand/sales/policy RNG',
                      'frozen DEV evaluation and forbidden final CLI', 'resume/logging exact parity',
                      'hash/source/disk/path guards and unchanged Phase1/V3 inputs'],
    }



### `agent02_assignment.phase02`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [51]:
%%inline_module agent02_assignment.phase02 agent02_assignment/phase02.py both
"""Bounded Phase 2 smoke only; no full-training or final-evaluation entry point."""

import argparse

import csv

import io

import json

import shutil

import time

from dataclasses import replace

from datetime import UTC, datetime

from pathlib import Path

import numpy as np

from .code.controlled import LearningConfig, MarketConfig

from .code.evaluation.frozen import evaluate_dev

from .code.reference_adapter import reference_smoke

from .code.training import TrainSession, digest, q_digest

from .phase01 import preflight

from .protocol import build_protocol

from .workspace import inside, sha256_file, write_receipt

BASE = Path(__file__).resolve().parent

SMOKE = 'evidence/phase02/smoke'

SOURCE_FILES = ('code/__init__.py', 'code/controlled.py', 'code/training.py',
                'code/evaluation/__init__.py', 'code/evaluation/frozen.py',
                'code/reference_adapter.py', 'protocol.py', 'phase02.py', 'workspace.py')



#### disk_guard

In [52]:
%%inline_module agent02_assignment.phase02 agent02_assignment/phase02.py both
def disk_guard(base):
    free = shutil.disk_usage(base)[2]
    if free < 5 * 1024**3:
        raise RuntimeError('Disk reserve requires at least 5 GiB; do not delete or alter system settings')
    return free



#### save_json

In [53]:
%%inline_module agent02_assignment.phase02 agent02_assignment/phase02.py both
def save_json(base, relative, value):
    # Validate finite values before creating a file; workspace helper enforces paths/no overwrite.
    normalized = json.loads(json.dumps(value, allow_nan=False))
    return write_receipt(base, relative, normalized)



#### save_csv

In [54]:
%%inline_module agent02_assignment.phase02 agent02_assignment/phase02.py both
def save_csv(base, relative, rows):
    target = inside(base, relative)
    stream = io.StringIO(newline='')
    fields = list(rows[0])
    writer = csv.DictWriter(stream, fieldnames=fields, lineterminator='\n')
    writer.writeheader()
    writer.writerows({key: json.dumps(value) if isinstance(value, (list, dict)) else value
                     for key, value in row.items()} for row in rows)
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open('x', encoding='utf-8', newline='') as output:
        output.write(stream.getvalue())
    return target



#### verify_smoke

In [55]:
%%inline_module agent02_assignment.phase02 agent02_assignment/phase02.py both
def verify_smoke(base):
    receipt = json.loads(inside(base, SMOKE + '/complete.json').read_text())
    for item in receipt['files']:
        target = inside(base, item['path'])
        if sha256_file(target) != item['sha256'] or target.stat().st_size != item['bytes']:
            raise ValueError('Smoke artifact hash/size mismatch: ' + item['path'])
    for item in receipt['source_files']:
        if sha256_file(BASE / item['path']) != item['sha256']:
            raise ValueError('Smoke source hash mismatch: ' + item['path'])
    return receipt



#### execute_smoke

In [56]:
%%inline_module agent02_assignment.phase02 agent02_assignment/phase02.py both
def execute_smoke(base, protocol):
    fingerprint = digest(protocol)
    if inside(base, SMOKE + '/complete.json').exists():
        receipt = verify_smoke(base)
        if receipt['protocol_sha256'] != fingerprint:
            raise ValueError('Completed smoke belongs to a different protocol')
        return receipt
    claim = inside(base, SMOKE + '/claim.json')
    if claim.exists():
        raise FileExistsError('Existing smoke claim without completion; preserve and investigate, never replay')
    free_before = disk_guard(base)
    # Exclusive creation prevents two writers from running the same smoke.
    save_json(base, SMOKE + '/claim.json', {'status': 'claimed', 'protocol_sha256': fingerprint,
              'started_utc': datetime.now(UTC).isoformat(), 'purpose': 'SMOKE_NOT_REPORT'})
    smoke = protocol['smoke']
    episodes = smoke['episodes_per_track']
    reference = reference_smoke(episodes=episodes, seed=smoke['reference_seed'])
    disk_guard(base)
    save_json(base, SMOKE + '/reference/receipt.json', reference)
    save_csv(base, SMOKE + '/reference/episode_rewards.csv', reference['rows'])
    save_json(base, SMOKE + '/reference/q_table.json', reference['q'])
    sources = [{'path': p, 'sha256': sha256_file(BASE / p)} for p in SOURCE_FILES]
    save_json(base, SMOKE + '/reference/manifest.json', {
        'run_id': f'phase02_smoke/REF/s{smoke["reference_seed"]}', 'purpose': 'SMOKE_NOT_REPORT',
        'seed': smoke['reference_seed'], 'episodes': episodes,
        'reference_sha256': reference['reference_sha256'], 'protocol_sha256': fingerprint})
    learning = replace(LearningConfig(), episodes=episodes)
    run = TrainSession(MarketConfig(), learning, smoke['controlled_seed'])
    started = time.perf_counter()
    try:
        run.run_chunk(episodes, trace_limit=20, guard=lambda: disk_guard(base))
    except RuntimeError:
        save_json(base, SMOKE + f'/controlled/paused_e{run.completed:06d}.json', run.checkpoint())
        raise
    controlled_seconds = time.perf_counter() - started
    saved = run.checkpoint()
    checkpoint_path = SMOKE + f'/controlled/checkpoint_e{run.completed:06d}.json'
    save_json(base, checkpoint_path, saved)
    restored = TrainSession.from_checkpoint(json.loads(inside(base, checkpoint_path).read_text()))
    if restored.checkpoint() != saved:
        raise ValueError('Serialized checkpoint did not roundtrip exactly')
    save_csv(base, SMOKE + '/controlled/episode_rewards.csv', run.rows)
    save_json(base, SMOKE + '/controlled/q_table.json', run.agent.q.tolist())
    policy = [{'state': [d, other], 'greedy_action_set': np.flatnonzero(np.isclose(
               run.agent.q[d, other], run.agent.q[d, other].max())).tolist()}
              for d in (0, 1) for other in (0, 1)]
    save_json(base, SMOKE + '/controlled/policy.json', policy)
    save_json(base, SMOKE + '/controlled/visits.json', run.visits.tolist())
    save_json(base, SMOKE + '/controlled/trace_sample.json', run.trace)
    save_json(base, SMOKE + '/controlled/manifest.json', {
        'run_id': f'phase02_smoke/CONTROLLED/s{run.seed}', 'purpose': 'SMOKE_NOT_REPORT',
        'config': {'market': saved['market'], 'learning': saved['learning'], 'seed': run.seed},
        'config_sha256': digest({'market': saved['market'], 'learning': saved['learning'], 'seed': run.seed}),
        'protocol_sha256': fingerprint, 'source_files': sources,
        'checkpoint_path': checkpoint_path, 'checkpoint_payload_sha256': saved['sha256']})
    before = run.checkpoint()
    dev = [evaluate_dev(run.agent.q, run.market, episodes=smoke['development_episodes_per_role'],
                       scenario_seed=smoke['development_scenario_seed'], role=role) for role in (0, 1)]
    if before != run.checkpoint():
        raise ValueError('DEV evaluation changed training state or RNG')
    save_json(base, SMOKE + '/development.json', {'purpose': 'frozen_DEV_smoke_only', 'roles': dev})
    paths = sorted(path for path in inside(base, SMOKE).rglob('*') if path.is_file())
    records = [{'path': path.relative_to(base).as_posix(), 'bytes': path.stat().st_size,
                'sha256': sha256_file(path)} for path in paths]
    receipt = {'status': 'PASS', 'purpose': 'SMOKE_NOT_REPORT', 'protocol_sha256': fingerprint,
               'episodes_per_track': episodes, 'training_market_steps': episodes * run.market.steps * 2,
               'development_market_steps': sum(item['steps'] for item in dev), 'full_training_started': False,
               'final_evaluation_started': False, 'reference_main_called': False, 'charts_generated': 0,
               'checkpoint_roundtrip_exact': True, 'evaluation_training_state_unchanged': True,
               'controlled_q_sha256': q_digest(run.agent.q), 'files': records,
               'source_files': sources,
               'host_free_bytes_before': free_before, 'host_free_bytes_after': disk_guard(base),
               'wall_seconds': {'reference': reference['wall_seconds'], 'controlled': controlled_seconds},
               'artifact_bytes_excluding_complete': sum(item['bytes'] for item in records),
               'completed_utc': datetime.now(UTC).isoformat()}
    save_json(base, SMOKE + '/complete.json', receipt)
    return verify_smoke(base)



#### main

In [57]:
%%inline_module agent02_assignment.phase02 agent02_assignment/phase02.py both
def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('task', choices=('status', 'write-protocol', 'smoke', 'verify-smoke'))
    args = parser.parse_args()
    current = preflight()
    protocol = build_protocol()
    if args.task == 'write-protocol':
        save_json(BASE, 'docs/protocol_v1.json', protocol)
        result = {'status': 'PASS', 'path': 'docs/protocol_v1.json', 'sha256': digest(protocol)}
    elif args.task == 'smoke':
        saved_protocol = json.loads((BASE / 'docs/protocol_v1.json').read_text())
        if digest(saved_protocol) != digest(protocol):
            raise ValueError('Protocol differs; preserve existing registration and create a revision')
        result = execute_smoke(BASE, saved_protocol)
    elif args.task == 'verify-smoke':
        result = verify_smoke(BASE)
    else:
        result = {'status': current['status'], 'phase': 2, 'declared_runs': len(protocol['training_runs']),
                  'full_training_started': False, 'final_evaluation_started': False,
                  'smoke_complete': (BASE / SMOKE / 'complete.json').exists()}
    print(json.dumps(result, ensure_ascii=False, indent=2))



#### Thiết lập / điểm vào

In [58]:
%%inline_module agent02_assignment.phase02 agent02_assignment/phase02.py both
if __name__ == '__main__':
    main()



### `agent02_assignment.code.baseline_runs`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [59]:
%%inline_module agent02_assignment.code.baseline_runs agent02_assignment/code/baseline_runs.py both
"""Persist a registered baseline once; explicit resume from saved episode/RNG state."""

import json

import time

from dataclasses import asdict

from datetime import UTC, datetime

import numpy as np

from ..phase02 import disk_guard, save_csv, save_json

from ..workspace import inside, sha256_file

from .controlled import integer

from .reference_training import ReferenceSession

from .training import TrainSession, digest, q_digest



#### new_session

In [60]:
%%inline_module agent02_assignment.code.baseline_runs agent02_assignment/code/baseline_runs.py both
def new_session(track, market, learning, seed):
    if track not in ('REF', 'CONTROLLED'):
        raise ValueError('Unsupported baseline track')
    return (ReferenceSession if track == 'REF' else TrainSession)(market, learning, seed)



#### summarize

In [61]:
%%inline_module agent02_assignment.code.baseline_runs agent02_assignment/code/baseline_runs.py both
def summarize(rows, steps):
    rewards = np.array([[row['rl_episode_reward'], row['rule_episode_reward']] for row in rows])
    counts = np.array([row['action_counts'] for row in rows]).sum(axis=0)
    result = {'partition': 'training', 'episodes': len(rows), 'market_steps': len(rows)*steps,
              'rl_action_counts': counts[0].tolist(), 'rule_action_counts': counts[1].tolist()}
    for index, name in enumerate(('rl', 'rule')):
        values = rewards[:, index] / steps
        result.update({name+'_total_reward': float(rewards[:, index].sum()),
                       name+'_mean_reward_per_step': float(values.mean()),
                       name+'_first100_mean_per_step': float(values[:100].mean()),
                       name+'_last100_mean_per_step': float(values[-100:].mean()),
                       name+'_episode_mean_std_ddof1': float(values.std(ddof=1)) if len(values) > 1 else None})
    return result



#### verify_run

In [62]:
%%inline_module agent02_assignment.code.baseline_runs agent02_assignment/code/baseline_runs.py both
def verify_run(base, run_id):
    prefix = 'artifacts/runs/' + run_id
    result = json.loads(inside(base, prefix + '/complete.json').read_text())
    for item in result['files']:
        path = inside(base, item['path'])
        if sha256_file(path) != item['sha256'] or path.stat().st_size != item['bytes']:
            raise ValueError('Run artifact hash/size mismatch: ' + item['path'])
    return result



#### execute_run

In [63]:
%%inline_module agent02_assignment.code.baseline_runs agent02_assignment/code/baseline_runs.py both
def execute_run(base, spec, market, learning, declaration_sha256, *, resume=False, interval=100):
    integer(interval, 'checkpoint interval', 1)
    prefix = 'artifacts/runs/' + spec['run_id']
    folder = inside(base, prefix)
    registration = {'spec': spec, 'market': asdict(market), 'learning': asdict(learning),
                    'declaration_sha256': declaration_sha256}
    registration_hash = digest(registration)
    if (folder / 'complete.json').exists():
        result = verify_run(base, spec['run_id'])
        if result['registration_sha256'] != registration_hash:
            raise ValueError('Completed run registration changed')
        return result
    disk_guard(base)
    claim_path = folder / 'claim.json'
    if claim_path.exists():
        if not resume:
            raise FileExistsError('Incomplete claim requires explicit resume; do not replay')
        claim = json.loads(claim_path.read_text())
        if claim['registration_sha256'] != registration_hash:
            raise ValueError('Resume registration changed')
        checkpoints = sorted(folder.glob('checkpoint_e*.json'))
        saved = json.loads(checkpoints[-1].read_text())
        cls = ReferenceSession if spec['track'] == 'REF' else TrainSession
        run = cls.from_checkpoint(saved)
    else:
        run = new_session(spec['track'], market, learning, spec['seed'])
        save_json(base, prefix + '/claim.json', {'registration': registration,
                  'registration_sha256': registration_hash, 'started_utc': datetime.now(UTC).isoformat()})
        save_json(base, prefix + '/checkpoint_e000000.json', run.checkpoint())
    while run.completed < learning.episodes:
        start_episode = run.completed
        started, cpu_started = time.perf_counter(), time.process_time()
        status = 'COMPLETE'
        try:
            run.run_chunk(min(interval, learning.episodes-run.completed), trace_limit=20,
                          guard=lambda: disk_guard(base))
        except RuntimeError:
            status = 'PAUSED'
            raise
        finally:
            timing = {'start_episode': start_episode, 'end_episode': run.completed, 'status': status,
                      'train_wall_seconds': time.perf_counter()-started,
                      'train_cpu_seconds': time.process_time()-cpu_started}
            save_json(base, prefix + f'/checkpoint_e{run.completed:06d}.json', run.checkpoint())
            number = len(list(folder.glob('segment_*.json'))) + 1
            save_json(base, prefix + f'/segment_{number:04d}.json', timing)
    checkpoint = run.checkpoint()
    save_csv(base, prefix + '/episode_rewards.csv', run.rows)
    save_json(base, prefix + '/q_table.json', run.agent.q.tolist())
    save_json(base, prefix + '/visits.json', run.visits.tolist())
    save_json(base, prefix + '/trace_sample.json', run.trace)
    policy = [{'state': [d, other], 'q_no_discount': float(run.agent.q[d, other, 0]),
               'q_discount': float(run.agent.q[d, other, 1]), 'greedy_action_set': np.flatnonzero(
                   np.isclose(run.agent.q[d, other], run.agent.q[d, other].max())).tolist()}
              for d in (0, 1) for other in (0, 1)]
    save_json(base, prefix + '/policy.json', policy)
    summary = summarize(run.rows, market.steps)
    save_json(base, prefix + '/summary.json', summary)
    segments = [json.loads(path.read_text()) for path in sorted(folder.glob('segment_*.json'))]
    save_json(base, prefix + '/manifest.json', {**registration, 'registration_sha256': registration_hash,
              'device': 'NumPy CPU float64', 'checkpoint_payload_sha256': checkpoint['sha256'],
              'q_sha256': q_digest(run.agent.q), 'initialization': 'fresh zeros, not smoke checkpoint',
              'reference_execution': 'unchanged teacher classes, parity-tested instrumented loop'
              if spec['track'] == 'REF' else 'frozen Phase2 TrainSession'})
    paths = sorted(path for path in folder.rglob('*') if path.is_file())
    result = {'status': 'COMPLETE', 'run_id': spec['run_id'], 'registration_sha256': registration_hash,
              'summary': summary, 'q_sha256': q_digest(run.agent.q), 'resumed': resume,
              'train_wall_seconds': sum(row['train_wall_seconds'] for row in segments),
              'train_cpu_seconds': sum(row['train_cpu_seconds'] for row in segments),
              'files': [{'path': path.relative_to(base).as_posix(), 'sha256': sha256_file(path),
                         'bytes': path.stat().st_size} for path in paths]}
    save_json(base, prefix + '/complete.json', result)
    return verify_run(base, spec['run_id'])



### `agent02_assignment.code.augmented_runs`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [64]:
%%inline_module agent02_assignment.code.augmented_runs agent02_assignment/code/augmented_runs.py both
"""Persist preregistered augmented runs with explicit episode/RNG resume and immutable DONE."""

import json

import time

from dataclasses import asdict

from itertools import product

import numpy as np

from ..phase02 import disk_guard, save_csv, save_json

from ..workspace import inside, sha256_file

from .augmented import AugmentedSession, augmented_hash, state_index

from .baseline_runs import summarize, verify_run

from .controlled import integer

from .training import digest



#### execute_augmented

In [65]:
%%inline_module agent02_assignment.code.augmented_runs agent02_assignment/code/augmented_runs.py both
def execute_augmented(base, spec, market, learning, proof, *, interval=100, resume=False):
    integer(interval, 'interval', 1)
    prefix = 'artifacts/runs/'+spec['run_id']
    folder = inside(base, prefix)
    registration = {'spec': spec, 'market': asdict(market), 'learning': asdict(learning), 'declaration_sha256': proof}
    fingerprint = digest(registration)
    if (folder/'complete.json').exists():
        saved = verify_run(base, spec['run_id'])
        if saved['registration_sha256']!=fingerprint:
            raise ValueError('Completed augmented registration changed')
        return saved
    disk_guard(base)
    if (folder/'claim.json').exists():
        if not resume:
            raise FileExistsError('Incomplete augmented claim requires explicit resume')
        if json.loads((folder/'claim.json').read_text())['registration_sha256']!=fingerprint:
            raise ValueError('Resume registration changed')
        run = AugmentedSession.from_checkpoint(json.loads(max(folder.glob('checkpoint_e*.json')).read_text()))
    else:
        run = AugmentedSession(market, learning, spec['seed'])
        save_json(base, prefix+'/claim.json', {'registration': registration, 'registration_sha256': fingerprint})
        save_json(base, prefix+'/checkpoint_e000000.json', run.checkpoint())
    while run.completed<learning.episodes:
        start, wall, cpu, status = run.completed, time.perf_counter(), time.process_time(), 'COMPLETE'
        try:
            run.run_chunk(min(interval, learning.episodes-run.completed), trace_limit=20, guard=lambda: disk_guard(base))
        except BaseException:
            status = 'PAUSED'
            raise
        finally:
            timing = {'start_episode': start, 'end_episode': run.completed, 'status': status,
                'train_wall_seconds': time.perf_counter()-wall, 'train_cpu_seconds': time.process_time()-cpu}
            save_json(base, prefix+f'/checkpoint_e{run.completed:06d}.json', run.checkpoint())
            number = len(list(folder.glob('segment_*.json')))+1
            save_json(base, prefix+f'/segment_{number:04d}.json', timing)
    save_csv(base, prefix+'/episode_rewards.csv', run.rows)
    save_json(base, prefix+'/q_table.json', run.agent.q.tolist())
    save_json(base, prefix+'/visits.json', run.visits.tolist())
    save_json(base, prefix+'/trace_sample.json', run.trace)
    policies = []
    for d, own, other, remaining in product((0,1), (0,1), (0,1), range(1,21)):
        state = (d,own,other,remaining)
        values = run.agent.q[state_index(state)]
        policies.append({'state': list(state), 'q_values': values.tolist(),
            'greedy_actions': np.flatnonzero(np.isclose(values, values.max())).tolist(),
            'visits': run.visits[state_index(state)].tolist()})
    save_json(base, prefix+'/policy.json', policies)
    summary = {**summarize(run.rows, market.steps), 'q_entries': 320,
        'visited_state_action_cells': int(np.count_nonzero(run.visits)), 'q_payload_bytes_float64': int(run.agent.q.nbytes)}
    save_json(base, prefix+'/summary.json', summary)
    save_json(base, prefix+'/manifest.json', {**registration, 'device': 'NumPy CPU float64',
        'state_order': ['demand','own_previous','competitor_previous','remaining_steps'],
        'q_axis_order': ['demand','own_previous','competitor_previous','remaining_minus1','action'],
        'q_sha256': augmented_hash(run.agent.q), 'initialization': 'fresh zero Q, no oracle labels'})
    segments = [json.loads(p.read_text()) for p in sorted(folder.glob('segment_*.json'))]
    paths = sorted(p for p in folder.rglob('*') if p.is_file())
    result = {'status': 'COMPLETE', 'run_id': spec['run_id'], 'registration_sha256': fingerprint, 'summary': summary,
        'q_sha256': augmented_hash(run.agent.q), 'resumed': resume,
        'train_wall_seconds': sum(s['train_wall_seconds'] for s in segments),
        'train_cpu_seconds': sum(s['train_cpu_seconds'] for s in segments),
        'files': [{'path':p.relative_to(base).as_posix(), 'bytes':p.stat().st_size, 'sha256':sha256_file(p)} for p in paths]}
    save_json(base, prefix+'/complete.json', result)
    return verify_run(base, spec['run_id'])



### `agent02_assignment.code.baseline_analysis`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [66]:
%%inline_module agent02_assignment.code.baseline_analysis agent02_assignment/code/baseline_analysis.py both
"""Validate and summarize saved episode logs; no agents or model forward passes."""

import numpy as np

from .controlled import integer



#### moving_average

In [67]:
%%inline_module agent02_assignment.code.baseline_analysis agent02_assignment/code/baseline_analysis.py both
def moving_average(values, window):
    integer(window, 'window', 1)
    array = np.asarray(values, dtype=float)
    if array.ndim != 1 or len(array) < window or not np.isfinite(array).all():
        raise ValueError('Moving average requires a finite vector at least window long')
    return np.convolve(array, np.ones(window)/window, mode='valid')



#### validate_rows

In [68]:
%%inline_module agent02_assignment.code.baseline_analysis agent02_assignment/code/baseline_analysis.py both
def validate_rows(rows, *, episodes, steps):
    if len(rows) != episodes:
        raise ValueError('Wrong number of episode rows')
    for index, row in enumerate(rows):
        scalar_fields = ['rl_episode_reward', 'rule_episode_reward', 'rl_mean_reward_per_step',
                         'rule_mean_reward_per_step', 'epsilon', 'alpha']
        if row['episode'] != index+1 or not np.isfinite([row[key] for key in scalar_fields]).all():
            raise ValueError('Episode order or finite scalar contract failed')
        counts = np.asarray(row['action_counts'])
        if (counts.shape != (2, 2) or not np.issubdtype(counts.dtype, np.integer)
                or (counts < 0).any() or not (counts.sum(axis=1) == steps).all()):
            raise ValueError('Action count mismatch')
        for name in ('rl', 'rule'):
            if not np.isclose(row[name+'_episode_reward']/steps, row[name+'_mean_reward_per_step']):
                raise ValueError('Mean reward denominator mismatch')



#### analyze_rows

In [69]:
%%inline_module agent02_assignment.code.baseline_analysis agent02_assignment/code/baseline_analysis.py both
def analyze_rows(rows, *, steps=20, window=100):
    validate_rows(rows, episodes=len(rows), steps=steps)
    result = {'training_seeds_per_track': 1, 'episodes': len(rows), 'window': window,
              'unit': 'simulated profit per market step', 'uncertainty_band': None}
    for name in ('rl', 'rule'):
        raw = np.array([row[name+'_mean_reward_per_step'] for row in rows])
        smooth = moving_average(raw, window)
        result[name] = {'first_window_mean': float(smooth[0]), 'last_window_mean': float(smooth[-1]),
                        'window_min': float(smooth.min()), 'window_max': float(smooth.max()),
                        'raw_episode_mean_std_ddof1': float(raw.std(ddof=1)),
                        'last_window_minus_first': float(smooth[-1]-smooth[0])}
    return result



### `agent02_assignment.code.sweep_analysis`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [70]:
%%inline_module agent02_assignment.code.sweep_analysis agent02_assignment/code/sweep_analysis.py both
"""One-factor contracts and training-seed statistics; no model execution."""

import numpy as np

from .baseline_analysis import moving_average

CONFIG_IDS = ('baseline', 'alpha_low', 'alpha_high', 'epsilon_low_fixed', 'epsilon_high_fixed')



#### validate_factors

In [71]:
%%inline_module agent02_assignment.code.sweep_analysis agent02_assignment/code/sweep_analysis.py both
def validate_factors(protocol):
    configs = protocol['controlled_configs']
    if set(configs) != set(CONFIG_IDS):
        raise ValueError('Exactly five core configurations are required')
    baseline = configs['baseline']
    for name in CONFIG_IDS[1:]:
        changed = {key for key in baseline if baseline[key] != configs[name][key]}
        expected = {'alpha'} if name.startswith('alpha') else {'epsilon_start', 'epsilon_min', 'epsilon_decay'}
        if changed != expected or set(configs[name]) != set(baseline):
            raise ValueError('Sweep must change only its declared factor')



#### aggregate_curves

In [72]:
%%inline_module agent02_assignment.code.sweep_analysis agent02_assignment/code/sweep_analysis.py both
def aggregate_curves(values, *, window=100):
    array = np.asarray(values, dtype=float)
    if array.ndim != 2 or array.shape[0] != 5 or not np.isfinite(array).all():
        raise ValueError('Expected five finite training-seed trajectories')
    smooth = np.array([moving_average(row, window) for row in array])
    return {'episode': np.arange(window, array.shape[1]+1).tolist(), 'mean': smooth.mean(axis=0).tolist(),
            'std': smooth.std(axis=0, ddof=1).tolist(), 'n_seeds': 5, 'ddof': 1, 'window': window}



#### paired_deltas

In [73]:
%%inline_module agent02_assignment.code.sweep_analysis agent02_assignment/code/sweep_analysis.py both
def paired_deltas(baseline, changed):
    if set(baseline) != set(changed) or len(baseline) != 5:
        raise ValueError('Paired training comparisons require the same five seed IDs')
    seeds = sorted(baseline)
    deltas = np.array([changed[seed]-baseline[seed] for seed in seeds], dtype=float)
    if not np.isfinite(deltas).all():
        raise ValueError('Paired deltas must be finite')
    return {'seeds': seeds, 'deltas': deltas.tolist(), 'mean': float(deltas.mean()),
            'std': float(deltas.std(ddof=1)), 'n_seeds': 5, 'ddof': 1, 'partition': 'training'}



### `agent02_assignment.code.evaluation.aggregate_v1`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [74]:
%%inline_module agent02_assignment.code.evaluation.aggregate_v1 agent02_assignment/code/evaluation/aggregate_v1.py both
"""Aggregate saved final results: role-average first, then five trained-seed replicates."""

import numpy as np

from ..sweep_analysis import CONFIG_IDS

SEEDS = (11, 22, 33, 44, 55)



#### validate_result

In [75]:
%%inline_module agent02_assignment.code.evaluation.aggregate_v1 agent02_assignment/code/evaluation/aggregate_v1.py both
def validate_result(result):
    rows, steps = result['rows'], result['steps']
    if len(rows) != result['episodes'] or steps != result['episodes']*20:
        raise ValueError('Episode/step count differs')
    for prefix, name in [('learner', 'total_reward'), ('opponent', 'opponent_total_reward')]:
        total = sum(row[prefix+'_reward'] for row in rows)
        mean_key = 'mean_reward' if prefix == 'learner' else 'opponent_mean_reward'
        counts = np.array([row[prefix+'_action_counts'] for row in rows])
        if (not np.isfinite(total) or total != result[name] or total/steps != result[mean_key]
                or counts.shape != (len(rows), 2) or (counts < 0).any()
                or not (counts.sum(axis=1) == 20).all()
                or counts.sum(axis=0).tolist() != result[prefix+'_action_counts']):
            raise ValueError('Saved reward/action aggregation inconsistent')



#### role_average

In [76]:
%%inline_module agent02_assignment.code.evaluation.aggregate_v1 agent02_assignment/code/evaluation/aggregate_v1.py both
def role_average(records):
    if len(records) != 2 or {record['job']['role'] for record in records} != {0, 1}:
        raise ValueError('Require exactly both roles')
    steps = sum(record['result']['steps'] for record in records)
    target = sum(record['result']['total_reward'] for record in records)
    opponent = sum(record['result']['opponent_total_reward'] for record in records)
    return {'target_mean': target/steps, 'opponent_mean': opponent/steps,
            'target_total_sum_roles': target, 'opponent_total_sum_roles': opponent, 'steps_sum_roles': steps,
            'discount_rate': sum(record['result']['learner_action_counts'][1] for record in records)/steps,
            'episodes_per_role': records[0]['result']['episodes'], 'roles': 2}



#### mean_std

In [77]:
%%inline_module agent02_assignment.code.evaluation.aggregate_v1 agent02_assignment/code/evaluation/aggregate_v1.py both
def mean_std(values):
    array = np.array(values, dtype=float)
    if len(array) != 5 or not np.isfinite(array).all():
        raise ValueError('Five finite seed-level values required')
    return float(array.mean()), float(array.std(ddof=1))



#### aggregate_evaluations

In [78]:
%%inline_module agent02_assignment.code.evaluation.aggregate_v1 agent02_assignment/code/evaluation/aggregate_v1.py both
def aggregate_evaluations(records):
    if len(records) != 71:
        raise ValueError('Require all 71 final claims')
    indexed = {}
    for record in records:
        validate_result(record['result'])
        job = record['job']
        key = (job['group'], job['config_id'], job['train_seed'], job['role'])
        if key in indexed:
            raise ValueError('Duplicate final job identity')
        indexed[key] = record
    def pair(group, config, seed):
        try:
            return role_average([indexed[group, config, seed, role] for role in (0, 1)])
        except KeyError as error:
            raise ValueError('Missing configuration/seed/role') from error
    controls = {(kind, seed): pair('control', kind, seed) for kind in ('rule', 'no_discount') for seed in SEEDS}
    multi = []
    for config in CONFIG_IDS:
        for seed in SEEDS:
            values = pair('core', config, seed)
            multi.append({'config_id': config, 'train_seed': seed, 'partition': 'final', **values,
                'within_match_delta': values['target_mean']-values['opponent_mean'],
                'rule_control_mean': controls['rule', seed]['target_mean'],
                'no_discount_control_mean': controls['no_discount', seed]['target_mean'],
                'delta_vs_rule_policy': values['target_mean']-controls['rule', seed]['target_mean'],
                'delta_vs_no_discount_policy': values['target_mean']-controls['no_discount', seed]['target_mean']})
    baseline = {row['train_seed']: row['target_mean'] for row in multi if row['config_id'] == 'baseline'}
    paired_rows, summaries = [], []
    for config in CONFIG_IDS:
        group = [row for row in multi if row['config_id'] == config]
        result = {'config_id': config, 'n_training_seeds': 5, 'roles_averaged_first': True, 'std_ddof': 1}
        for field in ('target_mean', 'opponent_mean', 'within_match_delta', 'delta_vs_rule_policy',
                      'delta_vs_no_discount_policy', 'discount_rate'):
            mean, std = mean_std([row[field] for row in group])
            stem = field.removesuffix('_mean')
            result[stem+'_mean'], result[stem+'_std'] = mean, std
        deltas = []
        for row in group:
            delta = row['target_mean']-baseline[row['train_seed']]
            deltas.append(delta)
            paired_rows.append({'config_id': config, 'train_seed': row['train_seed'],
                                'delta_vs_baseline': delta, 'delta_vs_rule_policy': row['delta_vs_rule_policy'],
                                'delta_vs_no_discount_policy': row['delta_vs_no_discount_policy'],
                                'within_match_delta': row['within_match_delta']})
        result['paired_delta_baseline_mean'], result['paired_delta_baseline_std'] = mean_std(deltas)
        summaries.append(result)
    reference = indexed.get(('reference_native', 'reference_native', 42, 0))
    if reference is None:
        raise ValueError('Missing native reference')
    native = reference['result']
    return {'multi_seed': multi, 'config_summary': summaries, 'paired_rows': paired_rows,
            'controls': [{'kind': kind, 'scenario_panel': seed, **controls[kind, seed]}
                         for kind in ('rule', 'no_discount') for seed in SEEDS],
            'reference_native': {'target_mean': native['mean_reward'], 'opponent_mean': native['opponent_mean_reward'],
                'target_total': native['total_reward'], 'opponent_total': native['opponent_total_reward'],
                'steps': native['steps'], 'roles': 1, 'n_training_seeds': 1, 'native_rng': True}}



### `agent02_assignment.code.gold_regime`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [79]:
%%inline_module agent02_assignment.code.gold_regime agent02_assignment/code/gold_regime.py both
"""Causal calendar-adjacent TRAIN-only price direction, used solely as a demand proxy."""

import re

from itertools import pairwise

import numpy as np



#### month_number

In [80]:
%%inline_module agent02_assignment.code.gold_regime agent02_assignment/code/gold_regime.py both
def month_number(value):
    match = re.fullmatch(r'(\d{4})[-M](\d{2})', str(value))
    if not match or not 1 <= int(match[2]) <= 12:
        raise ValueError('Expected YYYY-MM or YYYYMmm calendar month')
    return int(match[1])*12+int(match[2])-1



#### month_name

In [81]:
%%inline_module agent02_assignment.code.gold_regime agent02_assignment/code/gold_regime.py both
def month_name(number):
    return f'{number//12:04d}-{number%12+1:02d}'



#### prefix_records

In [82]:
%%inline_module agent02_assignment.code.gold_regime agent02_assignment/code/gold_regime.py both
def prefix_records(rows, column, cutoff):
    """Stop on first later calendar label before accessing its price field."""
    result = []
    for row in rows:
        if not re.fullmatch(r'\d{4}M\d{2}', str(row[0])):
            continue
        number = month_number(row[0])
        if number > month_number(cutoff):
            break
        result.append((month_name(number), row[column]))
    return result



#### estimate_regimes

In [83]:
%%inline_module agent02_assignment.code.gold_regime agent02_assignment/code/gold_regime.py both
def estimate_regimes(rows):
    if len(rows) < 2:
        raise ValueError('Require a nontrivial TRAIN history')
    numbers = [month_number(row[0]) for row in rows]
    if any(b <= a for a, b in pairwise(numbers)):
        raise ValueError('Months must be unique and strictly increasing')
    valid = [isinstance(price, (int, float)) and not isinstance(price, bool)
             and np.isfinite(price) and price > 0 for _, price in rows]
    regimes, counts = [], np.zeros((2, 2), dtype=np.int64)
    for i in range(1, len(rows)):
        if numbers[i] == numbers[i-1]+1 and valid[i] and valid[i-1]:
            regimes.append({'month': rows[i][0], 'regime': int(rows[i][1] > rows[i-1][1])})
    if not regimes:
        raise ValueError('No adjacent valid price pair')
    for previous, following in pairwise(regimes):
        if month_number(following['month']) == month_number(previous['month'])+1:
            counts[previous['regime'], following['regime']] += 1
    prices = np.array([p for (_, p), ok in zip(rows, valid, strict=True) if ok], dtype=float)
    missing = [month_name(n) for n in range(numbers[0], numbers[-1]+1) if n not in set(numbers)]
    return {'kernel': ((counts+1)/(counts.sum(axis=1, keepdims=True)+2)).tolist(),
            'transition_counts': counts.tolist(), 'smoothing_pseudocount': 1,
            'regime_counts': np.bincount([r['regime'] for r in regimes], minlength=2).tolist(),
            'regimes': regimes, 'records': [{'month': m, 'gold_usd_per_troy_ounce': p}
                                           for (m, p), ok in zip(rows, valid, strict=True) if ok],
            'first_month': rows[0][0], 'last_month': rows[-1][0], 'observed_months': len(rows),
            'valid_prices': int(sum(valid)), 'invalid_price_months': [m for (m, _), ok in zip(rows, valid, strict=True) if not ok],
            'missing_months': missing, 'gap_pairs_excluded': sum(b != a+1 for a, b in pairwise(numbers)),
            'price_summary': {'minimum': float(prices.min()), 'maximum': float(prices.max()),
                              'mean': float(prices.mean()), 'median': float(np.median(prices)),
                              'std_ddof1': float(prices.std(ddof=1)) if len(prices) > 1 else None},
            'real_demand_measurement': False, 'initial_demand': 'Bernoulli(0.5)',
            'interpretation': 'Historical price direction calibrates a synthetic demand kernel; no demand causality or investment validation'}



### `agent02_assignment.code.market_oracle`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [84]:
%%inline_module agent02_assignment.code.market_oracle agent02_assignment/code/market_oracle.py both
"""Finite-horizon known-model checks against fixed RuleAgent. Never supplies training labels."""

from itertools import product

import numpy as np

from .augmented import SHAPE, augmented_hash

from .controlled import RuleAgent, conversion, margin

from .training import q_digest



#### backward

In [85]:
%%inline_module agent02_assignment.code.market_oracle agent02_assignment/code/market_oracle.py both
def backward(market, gamma, choose):
    if not np.isfinite(gamma) or not 0<=gamma<=1 or market.steps>20:
        raise ValueError('Invalid oracle gamma or horizon')
    q = np.zeros(SHAPE)
    value = np.zeros((2,2,2,21))
    rule = RuleAgent()
    for remaining in range(1,market.steps+1):
        for d,own,other in product((0,1),repeat=3):
            opponent = rule.act((d,own))
            customers = market.customers_high if d else market.customers_low
            for action in (0,1):
                reward = customers*conversion(d,action,opponent)*margin(market,action)
                future = sum(market.kernel[d][following]*value[following,action,opponent,remaining-1] for following in (0,1))
                q[d,own,other,remaining-1,action] = reward+gamma*future
            values = q[d,own,other,remaining-1]
            actions = choose(d,own,other,remaining,values)
            value[d,own,other,remaining] = float(np.mean(values[actions]))
    return {'q':q,'value':value,'gamma':gamma,'reset_expected_return':float(value[:,0,0,market.steps].mean()),
            'scope':'known simulator/fixed rule/initial demandBernoulli0.5; exact expectation, not sampled final or learned oracle'}



#### solve

In [86]:
%%inline_module agent02_assignment.code.market_oracle agent02_assignment/code/market_oracle.py both
def solve(market, *, gamma=1.):
    return backward(market,gamma,lambda d,own,other,remaining,values: np.flatnonzero(
        np.isclose(values,values.max(),rtol=0,atol=1e-12)))



#### policy_value

In [87]:
%%inline_module agent02_assignment.code.market_oracle agent02_assignment/code/market_oracle.py both
def policy_value(market, *, q=None, kind='rule', gamma=1.):
    if kind not in ('rule','partial','augmented'):
        raise ValueError('Unknown oracle policy kind')
    if kind=='augmented':
        augmented_hash(q)
    elif kind=='partial':
        q_digest(q)
    array = np.asarray(q) if q is not None else None
    rule = RuleAgent()
    def choose(d,own,other,remaining,values):
        if kind=='rule':
            return [rule.act((d,other))]
        values = array[d,other] if kind=='partial' else array[d,own,other,remaining-1]
        return np.flatnonzero(np.isclose(values,values.max(),rtol=1e-5,atol=1e-8))
    return backward(market,gamma,choose)



### `agent02_assignment.code.pair_training`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [88]:
%%inline_module agent02_assignment.code.pair_training agent02_assignment/code/pair_training.py both
"""Two simultaneously learning tabular agents, independent policy RNGs and exact resume."""

import copy

import json

import time

from dataclasses import asdict

import numpy as np

from ..phase02 import disk_guard, save_csv, save_json

from ..workspace import inside, sha256_file

from .baseline_runs import verify_run

from .controlled import LearningConfig, MarketConfig, MarketEnv, QAgent, integer

from .training import digest, q_digest



#### PairSession

In [89]:
%%inline_module agent02_assignment.code.pair_training agent02_assignment/code/pair_training.py both
class PairSession:
    def __init__(self, market, learning, seed):
        self.market, self.learning, self.seed = market, learning, integer(seed, 'seed')
        self.env = MarketEnv(market, seed)
        self.agents = [QAgent(learning, int(np.random.SeedSequence([seed, channel]).generate_state(1)[0]))
                       for channel in (3, 4)]
        self.completed, self.rows, self.trace = 0, [], []
        self.visits = np.zeros((2, 2, 2, 2), dtype=np.int64)

    def run_chunk(self, episodes, *, trace_limit=0, guard=None):
        integer(episodes, 'chunk episodes', 1)
        integer(trace_limit, 'trace_limit')
        if self.completed+episodes > self.learning.episodes:
            raise ValueError('Chunk exceeds episode budget')
        for _ in range(episodes):
            if guard is not None:
                guard()
            states = self.env.reset()
            total, counts = np.zeros(2), np.zeros((2, 2), dtype=np.int64)
            epsilon = self.learning.epsilon_at(self.completed)
            for step in range(self.market.steps):
                actions = tuple(agent.act(states[i], epsilon) for i, agent in enumerate(self.agents))
                result = self.env.step(actions)
                for i, agent in enumerate(self.agents):
                    agent.learn(states[i], actions[i], result.rewards[i], result.states[i], result.done)
                    self.visits[(i, *states[i], actions[i])] += 1
                    counts[i, actions[i]] += 1
                total += result.rewards
                if len(self.trace) < trace_limit:
                    self.trace.append({'episode': self.completed+1, 'step': step+1,
                        'states_before': [list(s) for s in states], 'actions': list(actions),
                        'rewards': list(result.rewards), 'states_after': [list(s) for s in result.states],
                        'sales': list(result.sales), 'terminal': result.done})
                states = result.states
            self.completed += 1
            self.rows.append({'episode': self.completed, 'agent0_episode_reward': float(total[0]),
                'agent1_episode_reward': float(total[1]), 'agent0_mean_reward_per_step': float(total[0]/self.market.steps),
                'agent1_mean_reward_per_step': float(total[1]/self.market.steps),
                'epsilon': epsilon, 'alpha': self.learning.alpha, 'action_counts': counts.tolist()})

    def checkpoint(self):
        value = {'schema': 'ag02.pair.checkpoint.1', 'market': asdict(self.market), 'learning': asdict(self.learning),
            'seed': self.seed, 'completed': self.completed, 'q': [a.q.tolist() for a in self.agents],
            'visits': self.visits.tolist(), 'rows': self.rows, 'trace': self.trace,
            'rng': {'demand': self.env.demand_rng.bit_generator.state,
                'sales': [r.bit_generator.state for r in self.env.sales_rngs],
                'agents': [a.rng.bit_generator.state for a in self.agents]}}
        return copy.deepcopy({**value, 'sha256': digest(value)})

    @classmethod
    def from_checkpoint(cls, saved):
        value = copy.deepcopy(saved)
        fingerprint = value.pop('sha256')
        if value['schema'] != 'ag02.pair.checkpoint.1' or digest(value) != fingerprint:
            raise ValueError('Pair checkpoint schema/hash mismatch')
        result = cls(MarketConfig(**value['market']), LearningConfig(**value['learning']), value['seed'])
        result.completed = integer(value['completed'], 'completed')
        if result.completed > result.learning.episodes or len(value['rows']) != result.completed:
            raise ValueError('Checkpoint episode mismatch')
        for agent, q, rng in zip(result.agents, value['q'], value['rng']['agents'], strict=True):
            q_digest(q)
            agent.q[:] = q
            agent.rng.bit_generator.state = rng
        visits = np.asarray(value['visits'])
        if (visits.shape != (2, 2, 2, 2) or (visits < 0).any() or
                not np.all(visits.sum(axis=(1, 2, 3)) == result.completed*result.market.steps)):
            raise ValueError('Checkpoint visits mismatch')
        result.visits[:] = visits
        result.rows, result.trace = value['rows'], value['trace']
        result.env.demand_rng.bit_generator.state = value['rng']['demand']
        for rng, state in zip(result.env.sales_rngs, value['rng']['sales'], strict=True):
            rng.bit_generator.state = state
        return result



#### execute_pair

In [90]:
%%inline_module agent02_assignment.code.pair_training agent02_assignment/code/pair_training.py both
def execute_pair(base, spec, market, learning, proof, *, interval=100, resume=False):
    integer(interval, 'interval', 1)
    prefix = 'artifacts/runs/'+spec['run_id']
    folder = inside(base, prefix)
    registration = {'spec': spec, 'market': asdict(market), 'learning': asdict(learning), 'declaration_sha256': proof}
    fingerprint = digest(registration)
    if (folder/'complete.json').exists():
        result = verify_run(base, spec['run_id'])
        if result['registration_sha256'] != fingerprint:
            raise ValueError('Completed pair registration changed')
        return result
    disk_guard(base)
    if (folder/'claim.json').exists():
        if not resume:
            raise FileExistsError('Incomplete pair requires explicit resume')
        if json.loads((folder/'claim.json').read_text())['registration_sha256'] != fingerprint:
            raise ValueError('Resume registration changed')
        run = PairSession.from_checkpoint(json.loads(max(folder.glob('checkpoint_e*.json')).read_text()))
    else:
        run = PairSession(market, learning, spec['seed'])
        save_json(base, prefix+'/claim.json', {'registration': registration, 'registration_sha256': fingerprint})
        save_json(base, prefix+'/checkpoint_e000000.json', run.checkpoint())
    while run.completed < learning.episodes:
        start, wall, cpu = run.completed, time.perf_counter(), time.process_time()
        status = 'COMPLETE'
        try:
            run.run_chunk(min(interval, learning.episodes-run.completed), trace_limit=20, guard=lambda: disk_guard(base))
        except BaseException:
            status = 'PAUSED'
            raise
        finally:
            timing = {'start_episode': start, 'end_episode': run.completed, 'status': status,
                      'train_wall_seconds': time.perf_counter()-wall, 'train_cpu_seconds': time.process_time()-cpu}
            save_json(base, prefix+f'/checkpoint_e{run.completed:06d}.json', run.checkpoint())
            number = len(list(folder.glob('segment_*.json')))+1
            save_json(base, prefix+f'/segment_{number:04d}.json', timing)
    save_csv(base, prefix+'/episode_rewards.csv', run.rows)
    save_json(base, prefix+'/q_table.json', [a.q.tolist() for a in run.agents])
    save_json(base, prefix+'/visits.json', run.visits.tolist())
    save_json(base, prefix+'/trace_sample.json', run.trace)
    policy = [{'learner': i, 'state': [d, other], 'q_values': a.q[d, other].tolist(),
        'greedy_actions': np.flatnonzero(np.isclose(a.q[d, other], a.q[d, other].max())).tolist()}
        for i, a in enumerate(run.agents) for d in (0, 1) for other in (0, 1)]
    save_json(base, prefix+'/policy.json', policy)
    summary = {'episodes': run.completed, 'market_steps': run.completed*market.steps,
               'learners': 2, 'train_mean_per_step': [float(np.mean([r[f'agent{i}_mean_reward_per_step'] for r in run.rows]))
                                                   for i in (0, 1)]}
    save_json(base, prefix+'/summary.json', summary)
    save_json(base, prefix+'/manifest.json', {**registration, 'device': 'NumPy CPU float64',
        'initialization': 'fresh zero Q for both learners', 'policy_rng_channels': [3, 4]})
    segments = [json.loads(p.read_text()) for p in sorted(folder.glob('segment_*.json'))]
    paths = sorted(p for p in folder.rglob('*') if p.is_file())
    result = {'status': 'COMPLETE', 'run_id': spec['run_id'], 'registration_sha256': fingerprint,
        'summary': summary, 'q_sha256': [q_digest(a.q) for a in run.agents], 'resumed': resume,
        'train_wall_seconds': sum(s['train_wall_seconds'] for s in segments),
        'train_cpu_seconds': sum(s['train_cpu_seconds'] for s in segments),
        'files': [{'path': p.relative_to(base).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_file(p)} for p in paths]}
    save_json(base, prefix+'/complete.json', result)
    return verify_run(base, spec['run_id'])



### `agent02_assignment.code.phase06_analysis`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [91]:
%%inline_module agent02_assignment.code.phase06_analysis agent02_assignment/code/phase06_analysis.py both
"""Saved-output analysis: average physical roles, then dependent learners within a pair."""

import numpy as np

from .evaluation.aggregate_v1 import SEEDS, mean_std, role_average, validate_result



#### aggregate

In [92]:
%%inline_module agent02_assignment.code.phase06_analysis agent02_assignment/code/phase06_analysis.py both
def aggregate(records):
    if len(records) != 160:
        raise ValueError('Require160 Phase6 claims')
    indexed = {}
    for record in records:
        validate_result(record['result'])
        j = record['job']
        key = (j['group'], j['kernel_id'], j['policy_id'], j['seed'], j['role'])
        if key in indexed or j['seed'] not in SEEDS:
            raise ValueError('Duplicate identity or unexpected seed')
        indexed[key] = record
    def pair(group, kernel, policy, seed):
        try:
            return role_average([indexed[group, kernel, policy, seed, role] for role in (0, 1)])
        except KeyError as error:
            raise ValueError('Missing role/group/policy/seed') from error
    variant, deltas, summary = [], [], []
    for kernel in ('persistent', 'gold'):
        for seed in SEEDS:
            values = {}
            for policy in ('baseline_iid', 'retrained', 'rule', 'no_discount'):
                values[policy] = pair('variant', kernel, policy, seed)
                variant.append({'kernel': kernel, 'policy': policy, 'seed': seed, **values[policy]})
            deltas.append({'kernel': kernel, 'seed': seed,
                'retrained_minus_baseline': values['retrained']['target_mean']-values['baseline_iid']['target_mean'],
                'retrained_minus_rule': values['retrained']['target_mean']-values['rule']['target_mean'],
                'baseline_minus_rule': values['baseline_iid']['target_mean']-values['rule']['target_mean']})
        for policy in ('baseline_iid', 'retrained', 'rule', 'no_discount'):
            rows = [r for r in variant if r['kernel']==kernel and r['policy']==policy]
            mean, std = mean_std([r['target_mean'] for r in rows])
            summary.append({'kernel': kernel, 'policy': policy, 'target_mean': mean, 'target_std': std,
                'n_seed_or_scenario_panels': 5, 'n_means_trained_replicates': policy not in ('rule', 'no_discount'),
                'std_ddof': 1})
    learners, pairs, q8summary = [], [], []
    for checkpoint in (0, 1000, 2500, 5000):
        for seed in SEEDS:
            pairvalues = []
            for learner in (0, 1):
                values = pair('q8', 'iid', f'learner{learner}_e{checkpoint:06d}', seed)
                learners.append({'checkpoint_episode': checkpoint, 'seed': seed, 'learner': learner, **values})
                pairvalues.append(values['target_mean'])
            pairs.append({'checkpoint_episode': checkpoint, 'seed': seed,
                          'target_mean': float(np.mean(pairvalues)), 'learner_gap_absolute': abs(pairvalues[1]-pairvalues[0])})
        selected = [r for r in pairs if r['checkpoint_episode']==checkpoint]
        mean, std = mean_std([r['target_mean'] for r in selected])
        q8summary.append({'checkpoint_episode': checkpoint, 'target_mean': mean, 'target_std': std,
                         'n_independent_pair_seeds': 5, 'roles_then_learners_averaged_first': True, 'std_ddof': 1})
    return {'variant_seed': variant, 'variant_deltas': deltas, 'variant_summary': summary,
            'q8_learner': learners, 'q8_pair': pairs, 'q8_summary': q8summary}



### `agent02_assignment.code.phase07_analysis`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [93]:
%%inline_module agent02_assignment.code.phase07_analysis agent02_assignment/code/phase07_analysis.py both
"""Three-kernel, three-policy role-first aggregates from saved final outputs."""

from .evaluation.aggregate_v1 import SEEDS, mean_std, role_average, validate_result

KERNELS=('iid','persistent','gold')

POLICIES=('augmented','baseline','rule')



#### aggregate

In [94]:
%%inline_module agent02_assignment.code.phase07_analysis agent02_assignment/code/phase07_analysis.py both
def aggregate(records):
    if len(records)!=90:
        raise ValueError('Require30new+60reused final claims')
    indexed={}
    for record in records:
        validate_result(record['result'])
        j=record['job']
        key=(j['kernel_id'],j['policy_id'],j['seed'],j['role'])
        if key in indexed or j['kernel_id'] not in KERNELS or j['policy_id'] not in POLICIES or j['seed'] not in SEEDS:
            raise ValueError('Duplicate/unknown final identity')
        indexed[key]=record
    def pair(kernel,policy,seed):
        try:
            return role_average([indexed[kernel,policy,seed,role] for role in (0,1)])
        except KeyError as error:
            raise ValueError('Missing paired role/policy/seed') from error
    seedrows,summary,deltas,delta_summary=[],[],[],[]
    for kernel in KERNELS:
        for policy in POLICIES:
            for seed in SEEDS:
                seedrows.append({'kernel':kernel,'policy':policy,'seed':seed,**pair(kernel,policy,seed)})
            selected=[r for r in seedrows if r['kernel']==kernel and r['policy']==policy]
            mean,std=mean_std([r['target_mean'] for r in selected])
            rate_mean,rate_std=mean_std([r['discount_rate'] for r in selected])
            summary.append({'kernel':kernel,'policy':policy,'profit_mean':mean,'profit_std':std,
                'discount_rate_mean':rate_mean,'discount_rate_std':rate_std,'n_seed_or_panels':5,
                'trained_replicates':policy!='rule','std_ddof':1})
        for seed in SEEDS:
            new=pair(kernel,'augmented',seed)['target_mean']
            deltas.append({'kernel':kernel,'seed':seed,'delta_vs_baseline':new-pair(kernel,'baseline',seed)['target_mean'],
                          'delta_vs_rule':new-pair(kernel,'rule',seed)['target_mean']})
        row={'kernel':kernel,'n_training_seeds':5,'roles_averaged_first':True,'std_ddof':1}
        for field in ('delta_vs_baseline','delta_vs_rule'):
            mean,std=mean_std([r[field] for r in deltas if r['kernel']==kernel])
            row[field+'_mean'],row[field+'_std']=mean,std
        delta_summary.append(row)
    return {'seed_comparison':seedrows,'policy_summary':summary,'paired_deltas':deltas,'delta_summary':delta_summary}



### `agent02_assignment.code.report_assets`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [95]:
%%inline_module agent02_assignment.code.report_assets agent02_assignment/code/report_assets.py both
"""Pure helpers for source-faithful report assembly; no scientific runner imports."""

import ast

import hashlib

import math

import re

import textwrap

from pathlib import Path



#### decimal_vi

In [96]:
%%inline_module agent02_assignment.code.report_assets agent02_assignment/code/report_assets.py both
def decimal_vi(value, digits=5, *, group=False):
    number=float(value)
    if not math.isfinite(number):
        raise ValueError('Report numbers must be finite')
    formatted=format(number,(',' if group else '')+f'.{digits}f')
    return formatted.replace(',','_').replace('.',',').replace('_','.')



#### markdown_table

In [97]:
%%inline_module agent02_assignment.code.report_assets agent02_assignment/code/report_assets.py both
def markdown_table(headers, rows):
    def cells(values):
        return '|'+ '|'.join(str(v).replace('|','\\|').replace('\n',' ') for v in values)+'|'
    if any(len(row)!=len(headers) for row in rows):
        raise ValueError('Table row width differs')
    return '\n'.join([cells(headers),cells(['---']*len(headers)),*[cells(row) for row in rows]])



#### render_tokens

In [98]:
%%inline_module agent02_assignment.code.report_assets agent02_assignment/code/report_assets.py both
def render_tokens(template, values):
    used=[]
    def replace(match):
        key=match[1]
        if key not in values:
            raise ValueError('Unknown report token:'+key)
        used.append(key)
        return str(values[key])
    result=re.sub(r'\{\{([\w.:-]+)\}\}',replace,template)
    if '{{' in result or '}}' in result:
        raise ValueError('Unresolved/nested report token')
    return result,used



#### readable_prose

In [99]:
%%inline_module agent02_assignment.code.report_assets agent02_assignment/code/report_assets.py both
def readable_prose(text):
    """Add missing prose spaces while preserving code, IDs, paths and template tokens."""
    protected=(r'(```[\s\S]*?```|`[^`\n]+`|\{\{[^}]+\}\}|'
        r'\b(?:[A-Z]\d{2}[A-Z]+\d+|(?:Agent|AGENT|AG)\d+|REF\d+|CONTROLLED\d+|[FSDBE]\d{1,2}|Q(?:0?[0-9]|10)|float64|SHA256|PCG64|\d{4}\.\d{4,5}(?:v\d+)?)\b|'
        r'https?://\S+|\b(?:sources|artifacts|evidence|report|code|data|docs)/[A-Za-z0-9_./-]+)')
    parts=re.split(protected,text)
    for i in range(0,len(parts),2):
        value=parts[i]
        value=re.sub(r'(?<=[^\W\d_])(?=\d)',' ',value)
        value=re.sub(r'(?<=\d)(?=[^\W\d_])',' ',value)
        value=value.replace('×',' × ').replace('→',' → ')
        for old,new in [('Xepisode','X là episode'),('Xcấu','X là cấu'),('Xnămcấu','X là năm cấu'),
                        ('Xnăm','X là năm'),('Yprofit','Y là profit'),('Ymeanprofit','Y là mean profit'),
                        ('Ytỷ','Y là tỷ'),('Yxác','Y là xác'),('Ychênhprofit','Y là chênh profit'),
                        ('whiskerstd','whisker là std'),('meanprofit','mean profit'),('fixedrule','fixed rule')]:
            value=value.replace(old,new)
        parts[i]=re.sub(r' {2,}',' ',value)
    return ''.join(parts)



#### extract_code

In [100]:
%%inline_module agent02_assignment.code.report_assets agent02_assignment/code/report_assets.py both
def extract_code(base, relative, symbol, *, start=None, end=None):
    root=Path(base).resolve()
    target=(root/relative).resolve()
    if not target.is_relative_to(root):
        raise ValueError('Source path escapes assignment')
    raw=target.read_bytes()
    source=raw.decode('utf-8')
    nodes=ast.parse(source).body
    selected=None
    for name in symbol.split('.'):
        selected=next((node for node in nodes if getattr(node,'name',None)==name),None)
        if selected is None:
            raise ValueError('Source symbol missing:'+symbol)
        nodes=getattr(selected,'body',[])
    lines=source.splitlines()
    first,last=selected.lineno,selected.end_lineno
    fragment=lines[first-1:last]
    if start is not None:
        matches=[i for i,line in enumerate(fragment) if line==start]
        if len(matches)!=1:
            raise ValueError('Excerpt start must occur exactly once')
        first+=matches[0]
    if end is not None:
        matches=[i for i,line in enumerate(fragment) if line==end]
        if len(matches)!=1:
            raise ValueError('Excerpt end must occur exactly once')
        last=selected.lineno+matches[0]
    if last<first:
        raise ValueError('Excerpt range reversed')
    text=textwrap.dedent('\n'.join(lines[first-1:last]))
    return {'path':relative,'symbol':symbol,'start_line':first,'end_line':last,'text':text,
        'source_sha256':hashlib.sha256(raw).hexdigest(),'excerpt_sha256':hashlib.sha256(text.encode()).hexdigest()}



#### validate_structure

In [101]:
%%inline_module agent02_assignment.code.report_assets agent02_assignment/code/report_assets.py both
def validate_structure(text):
    sections=re.findall(r'^## ([1-9])\. ',text,re.MULTILINE)
    questions=re.findall(r'^### 6\.(\d+)\. Câu (\d+)\b',text,re.MULTILINE)
    figures=re.findall(r'!\[F(\d{2})\]',text)
    if sections!=[str(i) for i in range(1,10)]:
        raise ValueError('Require exactly nine ordered report sections')
    if questions!=[(str(i),str(i)) for i in range(1,11)]:
        raise ValueError('Require ten ordered question answers')
    if sorted(figures)!=[f'{i:02d}' for i in range(1,21)]:
        raise ValueError('Require twenty unique explained figures')
    if '{{' in text or '}}' in text or re.search(r'\b(TODO|TBD|PLACEHOLDER)\b',text):
        raise ValueError('No unresolved content placeholders')
    return {'sections':9,'questions':10,'figures':20}



### `agent02_assignment.code.sweep_runs`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [102]:
%%inline_module agent02_assignment.code.sweep_runs agent02_assignment/code/sweep_runs.py both
"""Select the preregistered 5x5 matrix and preserve the original baseline claim."""

from dataclasses import asdict

from .baseline_runs import execute_run, verify_run

from .controlled import LearningConfig, MarketConfig

from .sweep_analysis import CONFIG_IDS, validate_factors

from .training import digest

BASELINE_ID = 'controlled/iid/baseline/s0011'



#### select_specs

In [103]:
%%inline_module agent02_assignment.code.sweep_runs agent02_assignment/code/sweep_runs.py both
def select_specs(protocol):
    validate_factors(protocol)
    specs = [row for row in protocol['training_runs'] if row['track'] == 'CONTROLLED'
             and row['kernel_id'] == 'iid' and row['config_id'] in CONFIG_IDS]
    seeds = [11, 22, 33, 44, 55]
    expected = {(name, seed) for name in CONFIG_IDS for seed in seeds}
    actual = {(row['config_id'], row['seed']) for row in specs}
    if len(specs) != 25 or actual != expected or len({row['run_id'] for row in specs}) != 25:
        raise ValueError('Incomplete, duplicate or foreign core matrix cell')
    for row in specs:
        canonical = f'controlled/iid/{row["config_id"]}/s{row["seed"]:04d}'
        if (row['run_id'] != canonical or row['learners'] != 1
                or row['phase'] != (3 if canonical == BASELINE_ID else 4)):
            raise ValueError('Core cell identity or phase differs from registration')
    return sorted(specs, key=lambda row: (CONFIG_IDS.index(row['config_id']), row['seed']))



#### train_one

In [104]:
%%inline_module agent02_assignment.code.sweep_runs agent02_assignment/code/sweep_runs.py both
def train_one(base, spec, protocol, declaration_sha256, baseline_declaration_sha256, *, resume=False):
    market = MarketConfig(**protocol['market'])
    learning = LearningConfig(**protocol['controlled_configs'][spec['config_id']])
    reused = spec['run_id'] == BASELINE_ID
    if reused:
        result = verify_run(base, BASELINE_ID)
        registration = {'spec': spec, 'market': asdict(market), 'learning': asdict(learning),
                        'declaration_sha256': baseline_declaration_sha256}
        if result['registration_sha256'] != digest(registration):
            raise ValueError('Original Phase3 baseline registration changed')
    else:
        result = execute_run(base, spec, market, learning, declaration_sha256, resume=resume, interval=100)
    return {**result, 'reused_from_phase3': reused,
            'source_declaration_sha256': baseline_declaration_sha256 if reused else declaration_sha256}



## Thực hành: AG02_Code_and_Results

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

**Bản đường dẫn r2.** Notebook gốc và output được giữ trong provenance. Kernel: Python3.11 WSL NumPy/PyTorch. Không Run All pipeline huấn luyện chỉ để xem kết quả đã lưu.


# Agent02 — Code và kết quả thực nghiệm

Notebook đọc kết quả đã lưu của mô phỏng thương mại điện tử. Run All chỉ kiểm hash, trình bày code, bảng và hình; không train, cập nhật Q hoặc chạy đánh giá cuối. 56 market run/61 learner instance và 261 final claim thuộc Phase3–7. Các đoạn mô hình được hiển thị từ source đã khóa để học và đối chiếu, không thực thi. Ví dụ số học câu 3 là giả định của đề. Gold là proxy nhu cầu giả lập từ prefix TRAIN, không phải dữ liệu cầu hoặc kết quả đầu tư.

In [105]:
if RUN_EXPERIMENTS:
    import hashlib, json, platform
    from pathlib import Path
    from IPython.display import Code, Image, Markdown, display

    here = Path.cwd().resolve()
    candidates = [p for ancestor in [here, *here.parents] for p in (ancestor, ancestor / "agent02_assignment")]
    BASE = next(p for p in candidates if (p / "report/phase08/manuscript.md").is_file())
    def load(relative):
        return json.loads((BASE / relative).read_text(encoding="utf-8"))
    def sha(path):
        return hashlib.sha256(path.read_bytes()).hexdigest()
    inputs = load("report/phase08/input_index.json")
    before = {row["path"]: sha(BASE / row["path"]) for row in inputs}
    assert all(before[row["path"]] == row["sha256"] for row in inputs)
    print("Python:", platform.python_version())
    print("Input hashes PASS:", len(before), "| Device: CPU read-only | Training steps: 0 | Final forwards: 0")

## Cấu hình và phạm vi

Mỗi train run đã thực hiện 5.000 episode × 20 bước; seed CONTROLLED 11/22/33/44/55, REF42 riêng. Khi đo cuối, Q đóng băng, exploration tắt, 500 episode/vai và đổi vai. Gộp vai trước khi lấy mean/sample std qua năm seed; std không phải khoảng tin cậy. Hai learner trong cùng pair không phải hai replicate độc lập.

In [106]:
if RUN_EXPERIMENTS:
    protocol = load("docs/protocol_v1.json")
    display(Markdown("```json\n" + json.dumps(protocol, ensure_ascii=False, indent=2) + "\n```"))

## E01 — RuleAgent.act

RuleAgent.act chỉ giảm khi demand=1 và competitor_previous=1; không học từ reward.

Nguồn `code/controlled.py`, dòng 167–169; hiển thị nguyên đoạn, không thực thi.

In [107]:
if RUN_EXPERIMENTS:
    item = next(e for e in load("report/phase08/code_excerpt_index.json") if e["id"] == 'E01')
    assert sha(BASE / item["path"]) == item["source_sha256"]
    display(Code(item["text"], language="python"))
    print("Source SHA256:", item["source_sha256"], "| Excerpt SHA256:", item["excerpt_sha256"])

## E02 — QAgent.act

QAgent.act dùng epsilon-greedy khi train; tie-breaking có RNG riêng. Final tắt exploration.

Nguồn `code/controlled.py`, dòng 179–190; hiển thị nguyên đoạn, không thực thi.

In [108]:
if RUN_EXPERIMENTS:
    item = next(e for e in load("report/phase08/code_excerpt_index.json") if e["id"] == 'E02')
    assert sha(BASE / item["path"]) == item["source_sha256"]
    display(Code(item["text"], language="python"))
    print("Source SHA256:", item["source_sha256"], "| Excerpt SHA256:", item["excerpt_sha256"])

## E03 — QAgent.learn

QAgent.learn sửa một ô Q; terminal chỉ nhận reward, không bootstrap.

Nguồn `code/controlled.py`, dòng 192–200; hiển thị nguyên đoạn, không thực thi.

In [109]:
if RUN_EXPERIMENTS:
    item = next(e for e in load("report/phase08/code_excerpt_index.json") if e["id"] == 'E03')
    assert sha(BASE / item["path"]) == item["source_sha256"]
    display(Code(item["text"], language="python"))
    print("Source SHA256:", item["source_sha256"], "| Excerpt SHA256:", item["excerpt_sha256"])

## E04 — TrainSession.run_chunk

TrainSession.run_chunk chọn đồng thời hai action trước step, rồi learner mới cập nhật từ reward.

Nguồn `code/training.py`, dòng 46–52; hiển thị nguyên đoạn, không thực thi.

In [110]:
if RUN_EXPERIMENTS:
    item = next(e for e in load("report/phase08/code_excerpt_index.json") if e["id"] == 'E04')
    assert sha(BASE / item["path"]) == item["source_sha256"]
    display(Code(item["text"], language="python"))
    print("Source SHA256:", item["source_sha256"], "| Excerpt SHA256:", item["excerpt_sha256"])

## E05 — evaluate_tapes

evaluate_tapes đọc Q cố định, dùng tapes đã khai báo và không gọi learn.

Nguồn `code/evaluation/final_v1.py`, dòng 29–37; hiển thị nguyên đoạn, không thực thi.

In [111]:
if RUN_EXPERIMENTS:
    item = next(e for e in load("report/phase08/code_excerpt_index.json") if e["id"] == 'E05')
    assert sha(BASE / item["path"]) == item["source_sha256"]
    display(Code(item["text"], language="python"))
    print("Source SHA256:", item["source_sha256"], "| Excerpt SHA256:", item["excerpt_sha256"])

## E06 — evaluate_tapes

Phần cuối evaluator kiểm hash Q trước/sau để phát hiện cập nhật ngoài ý muốn.

Nguồn `code/evaluation/final_v1.py`, dòng 69–71; hiển thị nguyên đoạn, không thực thi.

In [112]:
if RUN_EXPERIMENTS:
    item = next(e for e in load("report/phase08/code_excerpt_index.json") if e["id"] == 'E06')
    assert sha(BASE / item["path"]) == item["source_sha256"]
    display(Code(item["text"], language="python"))
    print("Source SHA256:", item["source_sha256"], "| Excerpt SHA256:", item["excerpt_sha256"])

## E07 — markets

markets thay kernel IID bằng persistent/Gold; reward/action không đổi.

Nguồn `phase06.py`, dòng 43–47; hiển thị nguyên đoạn, không thực thi.

In [113]:
if RUN_EXPERIMENTS:
    item = next(e for e in load("report/phase08/code_excerpt_index.json") if e["id"] == 'E07')
    assert sha(BASE / item["path"]) == item["source_sha256"]
    display(Code(item["text"], language="python"))
    print("Source SHA256:", item["source_sha256"], "| Excerpt SHA256:", item["excerpt_sha256"])

## E08 — AugmentedAgent.learn

AugmentedAgent.learn thêm lịch sử của mình và thời gian; giảm remaining đúng một, giữ terminal.

Nguồn `code/augmented.py`, dòng 73–88; hiển thị nguyên đoạn, không thực thi.

In [114]:
if RUN_EXPERIMENTS:
    item = next(e for e in load("report/phase08/code_excerpt_index.json") if e["id"] == 'E08')
    assert sha(BASE / item["path"]) == item["source_sha256"]
    display(Code(item["text"], language="python"))
    print("Source SHA256:", item["source_sha256"], "| Excerpt SHA256:", item["excerpt_sha256"])

## Phép tính câu 3 và margin

Ví dụ giả định: Q cũ 4, alpha 0,1, reward 6, gamma 0,9, maxQ kế 5. Đây không phải kết quả của run đã train. Giá 10/chi phí 5 cho margin 5; giảm 20% cho margin 3.

In [115]:
if RUN_EXPERIMENTS:
    from decimal import Decimal as D
    target = D(6) + D("0.9") * D(5)
    delta = target - D(4)
    updated = D(4) + D("0.1") * delta
    assert updated == D("4.65")
    print("Target =", target, "| Delta =", delta, "| Q mới =", updated)
    print("Margin giữ giá =", D(10)-D(5), "| Margin giảm giá =", D(10)*D("0.8")-D(5))

## Bảng — alpha

Nguồn: `artifacts/metrics/phase04/config_train_summary.csv`; `artifacts/metrics/phase05/config_summary.json`. Số liệu hiển thị từ artifact; không chạy lại policy để lấy bảng.

In [116]:
if RUN_EXPERIMENTS:
    table = load("report/phase08/tables.json")['alpha']
    display(Markdown(table["markdown"]))

## Bảng — baseline_final

Nguồn: `artifacts/metrics/phase05/baseline_reward.csv`. Số liệu hiển thị từ artifact; không chạy lại policy để lấy bảng.

In [117]:
if RUN_EXPERIMENTS:
    table = load("report/phase08/tables.json")['baseline_final']
    display(Markdown(table["markdown"]))

## Bảng — baseline_train

Nguồn: `artifacts/metrics/phase03_train_summary.csv`. Số liệu hiển thị từ artifact; không chạy lại policy để lấy bảng.

In [118]:
if RUN_EXPERIMENTS:
    table = load("report/phase08/tables.json")['baseline_train']
    display(Markdown(table["markdown"]))

## Bảng — core_final

Nguồn: `artifacts/metrics/phase05/config_summary.json`. Số liệu hiển thị từ artifact; không chạy lại policy để lấy bảng.

In [119]:
if RUN_EXPERIMENTS:
    table = load("report/phase08/tables.json")['core_final']
    display(Markdown(table["markdown"]))

## Bảng — exploration

Nguồn: `artifacts/metrics/phase04/config_train_summary.csv`; `artifacts/metrics/phase05/config_summary.json`. Số liệu hiển thị từ artifact; không chạy lại policy để lấy bảng.

In [120]:
if RUN_EXPERIMENTS:
    table = load("report/phase08/tables.json")['exploration']
    display(Markdown(table["markdown"]))

## Bảng — gold_kernel

Nguồn: `data/derived/gold_regime_kernel_v1.json`. Số liệu hiển thị từ artifact; không chạy lại policy để lấy bảng.

In [121]:
if RUN_EXPERIMENTS:
    table = load("report/phase08/tables.json")['gold_kernel']
    display(Markdown(table["markdown"]))

## Bảng — improved

Nguồn: `artifacts/metrics/phase07/policy_summary.csv`. Số liệu hiển thị từ artifact; không chạy lại policy để lấy bảng.

In [122]:
if RUN_EXPERIMENTS:
    table = load("report/phase08/tables.json")['improved']
    display(Markdown(table["markdown"]))

## Bảng — improved_delta

Nguồn: `artifacts/metrics/phase07/delta_summary.csv`. Số liệu hiển thị từ artifact; không chạy lại policy để lấy bảng.

In [123]:
if RUN_EXPERIMENTS:
    table = load("report/phase08/tables.json")['improved_delta']
    display(Markdown(table["markdown"]))

## Bảng — oracle

Nguồn: `artifacts/metrics/phase07/oracle_summary.csv`; `artifacts/metrics/phase07/model_expectations.csv`. Số liệu hiển thị từ artifact; không chạy lại policy để lấy bảng.

In [124]:
if RUN_EXPERIMENTS:
    table = load("report/phase08/tables.json")['oracle']
    display(Markdown(table["markdown"]))

## Bảng — q8

Nguồn: `artifacts/metrics/phase06/q8_summary.csv`. Số liệu hiển thị từ artifact; không chạy lại policy để lấy bảng.

In [125]:
if RUN_EXPERIMENTS:
    table = load("report/phase08/tables.json")['q8']
    display(Markdown(table["markdown"]))

## Bảng — q_policy

Nguồn: `artifacts/runs/ref/baseline/s0042/policy.json`; `artifacts/runs/controlled/iid/baseline/s0011/policy.json`. Số liệu hiển thị từ artifact; không chạy lại policy để lấy bảng.

In [126]:
if RUN_EXPERIMENTS:
    table = load("report/phase08/tables.json")['q_policy']
    display(Markdown(table["markdown"]))

## Bảng — variants

Nguồn: `artifacts/metrics/phase06/variant_summary.csv`. Số liệu hiển thị từ artifact; không chạy lại policy để lấy bảng.

In [127]:
if RUN_EXPERIMENTS:
    table = load("report/phase08/tables.json")['variants']
    display(Markdown(table["markdown"]))

## Hình F01

Nguồn `artifacts/figures/phase03/ref_learning_curve.png`.

In [128]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F01')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** X là episode train, Y là reward cả episode chia 20 bước; đen liền là RL, xám đứt là rule. Panel dưới làm mượt 100 episode; chỉ có một seed, không có dải sai số giữa seed. RL tăng từ 31,92100 ở 100 episode đầu lên 33,56900 ở 100 episode cuối. Reward vẫn dao động; nhu cầu/doanh số ngẫu nhiên và policy đang học cùng ảnh hưởng đường vẽ. Làm mượt không chứng minh hội tụ hoặc ưu thế ngoài mẫu. Nguồn: D3, run REF42.

## Hình F02

Nguồn `artifacts/figures/phase03/controlled_learning_curve.png`.

In [129]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F02')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** X là episode train, Y là profit/bước; đen RL, xám đứt rule. Panel dưới là MA 100; chỉ một seed 11, không có dải sai số giữa seed và không phải các learner độc lập. RL đầu/cuối là 30,27150/ 34,31200, rule là 29,03400/ 33,36200. Rule không học vẫn thay reward do learner và thị trường đổi. Không lấy chênh giữa F01–F02 để xếp hạng cách quản lý RNG vì cả seed cũng khác. Nguồn: D3, run CONTROLLED11.

## Hình F03

Nguồn `artifacts/figures/phase03/q_tables.png`.

In [130]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F03')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** Hàng là bốn state nhu cầu/action trước của đối thủ; cột là hai action. Ô tối hơn có Q lớn hơn, số trong ô là Q chịu chiết khấu, không phải profit/bước. Không lấy trung bình hai track hoặc vẽ error bar. B02 cho thấy REF chọn 1 ởstate(0,1)/(1,1), trong khi CONTROLLED chọn 0 ởcảbốnstate. State ít hoặc không gặp trong policy greedy có thể không ảnh hưởng score của kịch bản hiện tại. Nguồn: D3, Q/policy cuối.

## Hình F04

Nguồn `artifacts/figures/phase03/action_counts.png`.

In [131]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F04')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** X là từng cặp agent/action; Y là tổng số lần chọn trên 100.000 bước/run. Cột tối giữ giá, cột sáng giảm giá; không có std vì mỗi track chỉ một seed. REF learner giảm 17524 lần, CONTROLLED giảm 14474 lần. Đây là hành vi khi còn exploration, không phải tỷ lệ epsilon hoặc tần suất của policy cuối. Giảm giá nhiều hơn không tự mang lợi nhuận cao hơn. Nguồn: D3, action_counts.

## Hình F05

Nguồn `artifacts/figures/phase04/alpha_curves.png`.

In [132]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F05')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** Mỗi panel ứng với α 0,01/0,10/0,50; X là episode, Y là profit/bước. Đen RL, xám đứt rule; MA 100 riêng từng seed trước rồi mean±sample std qua 5 seed. Cuối cửa sổ, RL lần lượt 33,49160,34,28750, 31,87860. Alpha lớn cập nhật mạnh nhưng không tạo final mean cao hơn trong mẫu này. Vùng mờ giữa seed khác với nhiễu từng bước; không dùng overlap của dải để làm kiểm định. Nguồn: D4, curves_mean_std.

## Hình F06

Nguồn `artifacts/figures/phase04/exploration_curves.png`.

In [133]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F06')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** X là episode, Y là profit/bước; đen RL, xám đứt rule. MA 100 riêng từng seed rồi mean/std qua 5 seed như F05; chỉ thay lịch khám phá. RL ở cửa sổ cuối của epsilon 0,05, baseline giảm, epsilon 0,30 lần lượt 32,57480, 34,28750, 31,15510. Khám phá cao có thể làm reward train thấp vì tiếp tục thử action, nhưng không kết luận chỉ từ train rằng policy cuối kém hơn. Nguồn: D4.

## Hình F07

Nguồn `artifacts/figures/phase04/epsilon_schedules.png`.

In [134]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F07')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** X là episode, Y là xác suất khám phá, không có đơn vị reward. Đường đen giảm 0,20 → 0,02; hai đường còn lại giữ 0,05/0,30. Đây là cấu hình chung cho cả 5 seed, không phải số đo ngẫu nhiên nên không có dải std. Epsilon chỉ là xác suất chọn action ngẫu nhiên; tỷ lệ giảm giá còn phụ thuộc action được chọn khi khám phá và policy khi khai thác. Nguồn: D2,D4, epsilon_schedules.

## Hình F08

Nguồn `artifacts/figures/phase04/train_summary.png`.

In [135]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F08')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** Hai panel tách RL và rule; X là cấu hình, Y là mean profit/bước trên 100.000 bước/run. Cột là mean 5 run, whisker là sample std của 5 mean/run. Baseline RL 33,76528±0,09989; alpha 0,01 30,91717±3,91720. Dải của alpha 0,01 lớn hơn nhiều giữa seed; không bỏ seed bất lợi. Đó là train score, chưa phải chất lượng cuối đo trước cùng opponent/tape. Nguồn: D4.

## Hình F09

Nguồn `artifacts/figures/phase05/final_profit.png`.

In [136]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F09')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** X là năm cấu hình,Y là profit/bước; cột mean, whisker là std qua 5 training seed sau gộp vai. Đường đứt là mean Rule-vs-rule qua 5 scenario panel, không phải rule đã train. Baseline 34,02786 cao nhất trong năm mean quan sát, alpha 0,50 là 32,06337; không từ đó chọn lại cấu hình hoặc khẳng định ưu thế thống kê/phổ quát. Nguồn: D5, config_summary.

## Hình F10

Nguồn `artifacts/figures/phase05/paired_and_match.png`.

In [137]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F10')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** Hai panel cùng X là cấu hình,Y là chênh profit/bước; chấm mean, whisker sample std qua 5 delta seed, đường 0 là không chênh. **Panel trái** là Q cấu hình trừ Q baseline cùng seed/tape; **panel phải** là Q trừ rule trong chính trận đó. Alpha 0,50 có delta so baseline -1,96449, nhưng delta trong trận 0,63457. Không dùng panel phải để tuyên bố Q tốt hơn policy rule trước cùng opponent. Nguồn: D5, paired_comparison.

## Hình F11

Nguồn `artifacts/figures/phase05/train_vs_final.png`.

In [138]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F11')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** X là cấu hình,Y là profit/bước; cột sáng là toàn quá trình học, cột tối là policy cuối đóng băng trên scenario riêng, whisker đều giữa 5 seed. Alpha 0,01 train mean 30,91717 còn finalmean 33,90382. Chênh lệch gồm cả policy thay đổi, exploration và tập tình huống; không quy toàn bộ cho việc tắt epsilon hoặc gọi score tăng là hội tụ. Nguồn: D4,D5.

## Hình F12

Nguồn `artifacts/figures/phase05/discount_rates.png`.

In [139]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F12')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** X là cấu hình,Y là tỷ lệ giảm giá(%); cột sáng train 100.000 bước/run, tối final 20.000 bước/seed gộp hai vai. Whisker là sample std giữa 5 seed, đơn vị điểm%. Final baseline 16,212%, alpha 0,50 30,186%. Tỷ lệ không bằng epsilon. Whisker mean±std có thể xuống dưới 0 khi độ phân tán lớn; từng tỷ lệ quan sát vẫn 0–100%, whisker không phải khoảng xác suất/CI. Nguồn: D4,D5.

## Hình F13

Nguồn `artifacts/figures/phase06/gold_calibration.png`.

In [140]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F13')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** Trái: X là tháng, Y là USD/troy ounce; đường đen là 560 giá tháng trong prefix TRAIN, 01/1960–08/2006. Phải: hàng là regime hiện tại, cột là regime tháng sau, số trong ô là xác suất sau cộng Laplace 1. Regime 1 là giá tăng; regime 0 gồm giảm hoặc giữ nguyên. P 00=0,706745, P 11=0,547511, khác kernel persistent giữ chế độ với xác suất 0,9. Đây là hướng biến động **giá**, không phải nhu cầu vàng thực đo; không diễn giải thành quan hệ nhân quả hay backtest đầu tư. Nguồn: S7,D6, Gold TRAIN prefix/kernel.

## Hình F14

Nguồn `artifacts/figures/phase06/two_learner_training.png`.

In [141]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F14')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** Hai panel tách learner 0/1; X là episode train, Y là profit/bước. Đường đen là mean của MA 100 riêng từng seed; dải xám là ±sample std qua năm pair seed. Hai đường vẫn dao động; mean train của learner 0/1 là 32,99454/ 32,98117, không tăng đều đến cuối. Đây là quá trình cả hai policy đang đổi; vùng mờ còn chứa khác biệt kinh nghiệm giữa seed. Không từ đó quy riêng biến động cho non-stationarity hoặc kết luận hội tụ. Nguồn: D6, pair_train_curves.

## Hình F15

Nguồn `artifacts/figures/phase06/checkpoint_drift.png`.

In [142]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F15')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** Trái: X là checkpoint, Y là profit/bước khi Q đóng băng trước fixed rule; chấm mean, whisker là std sau gộp vai/learner trong pair, n 5 pair seed. Mean ở 1.000 episode là 34,48443, ở 5.000 là 31,58872. Phải: X là cuối mỗi khoảng 100 episode, Y là số state đổi tập greedy trên bốn state, lấy mean qua 10 learner thuộc 5 pair. Có 612 state-change trên 500 learner-interval; không phải số lần đổi action trong timestep. Drift ở state ít gặp có thể không đổi return; không chọn checkpoint 1.000 làm kết quả cuối chỉ vì mean cao hơn. Nguồn: D6.

## Hình F16

Nguồn `artifacts/figures/phase06/variant_robustness.png`.

In [143]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F16')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** Mỗi panel là một kernel; X gồm Q IID cũ, Q train lại, rule và no-discount; Y là profit/bước. Cột mean, whisker là std sau gộp hai vai. Delta train lại trừ Q IID cũ là-1,48087±2,70026 ởpersistent, -0,80510±3,54276 ởGold. Rule-vs-rule và no-discount trùng hành vi giữ giá từ lịch sử reset 0, nên không coi hai cột đó là hai chuẩn hành vi độc lập. Không so raw profit xuyên kernel như cùng phân phối. Nguồn: D6, variant_seed/variant_summary/variant_deltas.

## Hình F17

Nguồn `artifacts/figures/phase07/augmented_training.png`.

In [144]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F17')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** Ba panel là IID/persistent/Gold; X là episode, Y là profit/bước. Đen liền là state memory/time, xám đứt là state cũ train cùng kernel; mỗi đường là mean MA 100 qua năm seed, dải là sample std tương ứng. Ở cả ba panel, đường memory/time quan sát thấp hơn phần lớn quá trình. Train reward còn exploration và Q thay đổi; không thay nó cho final hoặc suy ra nguyên nhân chỉ từ hình. Nguồn: D7, train_curves.

## Hình F18

Nguồn `artifacts/figures/phase07/augmented_final_profit.png`.

In [145]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F18')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** X là ba policy, Y là profit/bước; cột mean và whisker sample std, n 5 sau gộp vai. IID memory/time đạt 30,27341 so với baseline 34,02786; Gold lần lượt 27,10489/29,35694. Kết quả có std nhỏ hơn không đủ để gọi robust hơn nếu mean thấp hơn. Các panel khác phân phối nên so trong từng kernel là chính. Nguồn: D7, policy_summary.

## Hình F19

Nguồn `artifacts/figures/phase07/augmented_paired_delta.png`.

In [146]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F19')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** X là kernel, Y là profit augmented trừ đối chứng/bước. Tròn đen trừ baseline, vuông xám trừ rule; đường đứt 0 là không chênh. Điểm là mean, whisker là sample std của năm delta seed. Mean trừ baseline âm ở cả ba kernel: B11 ghi rõ mức chênh. Whisker qua 0 không phải kiểm định; không loại seed bất lợi hoặc tăng budget sau khi xem score để đảo kết luận. Nguồn: D7, paired_deltas/delta_summary.

## Hình F20

Nguồn `artifacts/figures/phase07/augmented_coverage_cost.png`.

In [147]:
if RUN_EXPERIMENTS:
    figure = next(f for f in load("report/phase08/figure_catalog.json") if f["id"] == 'F20')
    assert sha(BASE / figure["path"]) == figure["sha256"]
    display(Image(filename=str(BASE / figure["path"])))

**Giải thích:** X là kernel. Trái: Y là phần trăm ô state–action đã được thăm trên tổng ô lý thuyết; phải: Y là CPUgiây/100.000 train market step. Cột đậm memory/time, nhạt baseline; panel phải có whisker là std qua 5 seed. Agent mới thăm 83,6%/78,4%/ 81,8% của 320 ô trênIID/persistent/Gold, baseline thăm 100% của 8 ô; một số fullstate không đạt được từ reset/horizon nên phần chưa thăm không tự là lỗi. Qpayload float64 tăng 64 → 2.560 byte, không phải peakRAM. CPU loop không gồm checkpointI/O và không phải benchmark hệ thống. Nguồn: D7, cost_coverage.

## 6. Trả lời mười câu hỏi thí nghiệm

### 6.1. Câu 1 — Mô hình tác tử và môi trường

Hai tác tử là hai shop, dùng policy Q-learning và rule trong baseline. Môi trường
chung gồm nhu cầu, lịch sử action, cơ chế xác suất mua và reward. State local cũ
có bốn khả năng từ nhu cầu 0/1 và action trước của đối thủ 0/1; action giữ/giảm giá;
reward là profit. Sau action đồng thời, môi trường sinh sales, lưu action, lấy
nhu cầu mới và trả state/reward tiếp theo. E04 thể hiện đúng thứ tự này.

Theo PEAS, tiêu chí thành công là profit; môi trường là thị trường mô phỏng; bộ
chấp hành là quyết định giữ/giảm giá; thông tin quan sát là nhu cầu/lịch sử. Hai
policy cùng tác động đến xác suất mua và lịch sử của nhau nên đây là MAS qua môi
trường chung, dù không gửi thông điệp [S1, trang PDF 3–7; S6, trang PDF 3–5].

### 6.2. Câu 2 — Quyết định học từ kinh nghiệm và theo luật

| Khía cạnh | Rule-based | Q-learning |
|---|---|---|
| Ra quyết định | Theo nhánh điều kiện do người thiết kế viết | Theo Q đã học và epsilon khi train |
| Kinh nghiệm | Không tự sửa rule sau reward | Reward dùng sửa giá trị action |
| Thích nghi | Cần đổi rule nếu giả định thay đổi | Có thể học lại nhưng không bảo đảm tốt hơn |
| Chi phí | Không train; điều kiện đơn giản | Cần tương tác, lưu Q và kiểm tra nhiều seed |
| Giải thích | Chỉ ra nhánh luật đã dùng | Đọc state/action/Q và kết quả kiểm chứng |

Rule phù hợp khi ràng buộc rõ, cần quyết định nhất quán: ví dụ lọc nhà theo ngân
sách hoặc kiểm hồ sơ đặt lịch tư vấn tiểu đường có đủ trường thông tin. Ví dụ này
không đề xuất chẩn đoán/điều trị từ một nhãn dữ liệu. RL đáng cân nhắc khi quyết
định lặp lại có reward dài hạn và có môi trường thử nghiệm phù hợp, chẳng hạn
chính sách khuyến mãi trong simulator được kiểm chứng trước khi áp dụng.

Trong mẫu đang xét, rule control có mean 35,05235, cao hơn baseline
Q 34,02786; vì vậy khả năng học không đồng nghĩa ưu thế
mặc định. Kết quả này chỉ áp dụng simulator, budget và bộ seed hiện tại (B05,F09).

### 6.3. Câu 3 — Ý nghĩa cập nhật Q và phép tính cụ thể

Công thức cần dùng là:

**Q mới = Q cũ + α × [r + γ × max Q ở state kế − Q cũ].**

Q cũ là đánh giá đang có cho action vừa chọn. r là reward vừa nhận; maxQ ở state
kế là ước lượng tốt nhất cho bước tiếp theo; γ làm giảm trọng số lợi ích xa hơn.
Phần trong ngoặc là chênh giữa target mới và đánh giá cũ; α quyết định sửa bao
nhiêu phần của chênh đó. Ở terminal, target chỉ bằng r vì episode đã kết thúc.

Với số **giả định đề cho**, Q cũ 4, α 0,1, r 6, γ 0,9, maxQ kế 5:

1. Giá trị tương lai có chiết khấu:0,9 × 5=4,5.
2. Target:6+4,5=10,5.
3. Chênh target so Q cũ:10,5−4=6,5.
4. Phần cập nhật:0,1 × 6,5=0,65.
5. Q mới:4+0,65=**4,65**.

Q tăng từ 4 lên 4,65 vì tín hiệu mới tốt hơn đánh giá cũ, nhưng không nhảy thẳng lên
10,5 do alpha chỉ 0,1. Đây là phép tính giả định, không phải một reward hoặc Q đo
từ run. E03 và unit test kiểm đúng phép tính, đồng thời kiểm terminal. Công thức
theo [S1, trang PDF 6,10] và code; survey [S4] được dùng cho phần nền RL.

### 6.4. Câu 4 — Khám phá và khai thác

Epsilon-greedy chọn action ngẫu nhiên với xác suất epsilon; ngoài nhánh đó chọn Q
lớn nhất. Khám phá giúp thử action chưa biết rõ; khai thác dùng kinh nghiệm hiện
tại để thu reward. Khám phá quá nhiều có thể tiếp tục chọn action kém; dừng quá sớm
có thể giữ một đánh giá lệch vì ít được thử lại. Đây là đánh đổi, không phải quy
tắc “epsilon càng cao càng tốt”.

B04,F06–F07 so lịch giảm với epsilon 0,05 và 0,30 cố định. Mean train tương ứng
33,76528,32,89187,
30,92305; final tương ứng
34,02786,33,39952,
32,87013. Các số cuối đều đo với exploration
tắt, không phải vẫn giữ epsilon train. Lịch giảm có mean quan sát cao hơn trong
mẫu này, nhưng độ phân tán seed và n nhỏ hạn chế khẳng định tổng quát. E02 cho
thấy random tie vẫn khác exploration.

### 6.5. Câu 5 — Reward và mục tiêu kinh doanh

Không giảm giá: giá 10−chi phí 5=**5** lợi nhuận/món. Giảm 20%: giá mới 10 × 0,8=8,
lợi nhuận 8−5=**3**/món. Do đó sales tăng chưa chắc profit tổng tăng. Reward trong
bài ưu tiên profit, không đơn thuần số lượng bán [S1, trang PDF 4–5,10].

Nếu reward chỉ đếm sales, learner có thể thích action tăng xác suất mua dù margin
thấp. Nếu thêm tồn kho, phí khuyến mãi hoặc mục tiêu duy trì khách, hành vi tối ưu
cũng có thể khác. Đây là đề xuất thiết kế, không phải các reward đã được triển
khai trong số đo hiện tại. F04/F12 phản ánh action rate thực đo, không chứng minh
mối quan hệ nhân quả giữa giảm giá và profit ngoài mô hình.

### 6.6. Câu 6 — Tính reward trung bình và giới hạn một run

**Reward trung bình mỗi bước = tổng reward / tổng số bước.** B01 có ví dụ REF
train:3.345.548/100.000=33,45548 cho RL và
3.283.930/100.000=32,83930 cho rule. Đây là train
score, không phải kết quả policy cuối.

Với final native REF, tổng Q là 351275, rule
349685, mỗi shop 10000 bước;
mean tương ứng 35,12750/34,96850.
Native chỉ role 0 và RNG gốc, không đưa vào nhóm CONTROLLED n 5 (B06).

Baseline CONTROLLED sau gộp vai có Q mean 34,02786±
1,22178, rule trong cùng trận
33,21968±2,32629. Khi đối chứng
policy rule trước cùng fixed opponent, mean lại là 35,05235. Vì
đối thủ phản ứng với lịch sử, hai phép so sánh không được dùng thay nhau (B05,F10).

Một run chỉ quan sát một chuỗi kinh nghiệm. Nhu cầu, sales, exploration và phá
đồng hạng có thể làm Q cuối khác nhau giữa seed; B06 cho thấy các mean khác nhau
thật. Năm seed và sample std hữu ích hơn một run, nhưng không chứng minh ưu thế
ở thị trường thật hoặc coi mọi timestep là mẫu độc lập.

### 6.7. Câu 7 — Learning rate và độ ổn định

Alpha nhỏ chỉ sửa Q một phần nhỏ sau reward mới; alpha lớn phản ứng mạnh hơn với
quan sát gần nhất. Cập nhật mạnh có thể phản ứng nhanh nhưng cũng nhạy với nhiễu;
alpha nhỏ không tự bảo đảm policy tốt nếu budget hữu hạn và kinh nghiệm khác nhau.

B03,F05 đo α 0,01/0,10/0,50 với các phần khác giữ nguyên. Final mean±std lần lượt
33,90382±2,60833,
34,02786±1,22178,
32,06337±4,25085. Alpha 0,50
có độ phân tán final lớn nhất trong ba nhóm, trong khi std của mean **train** của
nó không lớn như vậy. Hai đại lượng đo khác nhau, nên không dùng curve train
ổn định hơn để kết luận policy cuối ổn định hơn. Kết luận giới hạn ở cấu hình,
budget và seed đã đo; không chọn lại alpha sau final.

### 6.8. Câu 8 — Tương tác và non-stationarity

Action hiện tại của đối thủ ảnh hưởng xác suất mua của shop mình; action trước
ảnh hưởng observation và phản ứng của rule ở bước sau. Khi hai Q-learning cùng
sửa policy, learner gặp một “đối thủ đang thay đổi”: cùng local observation/action
có thể cho kết quả khác do lựa chọn của đối thủ khác, ngoài nhiễu sales vốn có.
Điều này làm việc học khó hơn so với giả định đối thủ cố định.

Một thay đổi thực nghiệm là đóng băng checkpoint rồi đo trước cùng fixed opponent
và tapes. Bài đã thực hiện năm pair seed, hai learner riêng Q/RNG và bốn mốc 0/
1.000/2.500/5.000. B07,F15 có mean 34,48443 ởmốc 1.000 rồi
31,58872 ởmốc 5.000. Không có xu hướng final return tăng đều.
Policy drift và reward dao động chưa tách được riêng tác động nhân quả của đối
thủ đang học, vì alpha, observation và nhiễu vẫn tồn tại. Không chọn mốc tốt nhất
để thay checkpoint cuối hoặc xem 10 learner phụ thuộc là 10 replicate độc lập.

### 6.9. Câu 9 — Hạn chế và sửa môi trường thật sự

Ít nhất năm hạn chế là: conversion chỉ giả định minh họa; nhu cầu IID quá đơn giản;
state cũ thiếu lịch sử của mình/thời gian còn lại; không có inventory hoặc dị biệt
khách; hai shop không chia một lượng sales độc quyền. Gold bổ trợ vẫn là proxy
từ hướng giá, không phải dữ liệu cầu đo được. Các giả định này giới hạn khả năng
suy rộng, kể cả khi mean/std đã được tính đúng.

Sửa đổi đã triển khai là dùng demand Markov: persistent giữ chế độ với xác suất 0,9,
hoặc kernel hiệu chỉnh từ TRAIN Gold. Các phần giá/reward/action/rule/budget giữ
nguyên. E07 tạo các MarketConfig tương ứng; TrainSession cũ dùng đúng kernel đó,
không sửa thuật toán baseline để tạo kết quả thắng:

```python
def markets():
    data = protocol()
    baseline = MarketConfig(**data['market'])
    return {'iid': baseline, 'persistent': replace(baseline, kernel=data['future_variants']['persistent']['kernel']),
        'gold': replace(baseline, kernel=json.loads((BASE/'data/derived/gold_regime_kernel_v1.json').read_text())['kernel'])}
```

Nguồn E07: `phase06.py`, dòng 43–47; SHA256 `6cfec22472893221…` (hash đầy đủ trong code_excerpt_index.json).

Chỉ dùng 560 giá tháng đến 08/2006 để fit kernel (B08,F13). Trên mỗi
kernel mới, đo cả Q IID cũ đóng băng và Q train lại trước cùng fixed rule/tape.
Paired delta retrain−baseline ởpersistent là-1,48087,
ởGold-0,80510 (B09,F16). Trong budget này, train lại không
cải thiện mean. Không từ đó kết luận retrain luôn kém: từng seed khác nhau, chỉ
hai kịch bản sửa môi trường và n 5 nhỏ. Sửa kernel giúp kiểm độ bền trước thay đổi
nhu cầu giả lập, chưa giải quyết toàn bộ các hạn chế trên.

### 6.10. Câu 10 — Thiết kế agent đầy đủ hơn và kiểm chứng

Đề xuất thêm own_previous và remaining_steps vào state, giữ action giữ/giảm giá
và reward profit, dùng Q-learning 320 giá trị. Lý do là hai lịch sử cùng observation
cũ có thể làm rule hiện tại chọn khác nhau; thời gian còn lại cũng quyết định có
nên tính giá trị tương lai hay không. E08 cho thấy thời gian phải giảm đúng 1 và
terminal chỉ nhận reward hiện tại:

```python
def learn(self, state, action, reward, following, terminal):
    index = state_index(state)
    action = binary(action, 'action')
    if not np.isfinite(reward) or bool(terminal)!=(state[3]==1):
        raise ValueError('Invalid reward/terminal time')
    if terminal:
        target = reward
    else:
        following_index = state_index(following)
        if following[3]!=state[3]-1:
            raise ValueError('Remaining time must decrease exactly1')
        target = reward+self.config.gamma*float(self.q[following_index].max())
    key = (*index, action)
    self.q[key] += self.config.alpha*(target-self.q[key])
    if not np.isfinite(self.q[key]):
        raise ValueError('Nonfinite augmented Q update')
```

Nguồn E08: `code/augmented.py`, dòng 73–88; SHA256 `3638a130760b2bc3…` (hash đầy đủ trong code_excerpt_index.json).

Để kiểm robust, train trên ba kernel với cùng 5 seed và 5.000 × 20 budget, sau đó đo Q
đóng băng 500 episode/vai trước cùng fixed rule/tapes. So với baseline **đã train
trên cùng kernel** và rule; đọc paired return, std giữa seed, action rate và cost.
Các đối chứng cũ được reuse output, không forward lại để thay điểm (B10–B12,F17–F20).

Trong thực nghiệm, augmented−baseline mean là-3,75445
trênIID,-2,08854 trênpersistent và
-2,25205 trênGold. Mean đều thấp hơn đối chứng nên
chưa đạt mục tiêu cải thiện robust. State biểu diễn đầy đủ hơn là một lý do thiết
kế, không phải bảo đảm học tốt hơn dưới budget cố định. Bảng lớn hơn/cùng số tương
tác là một giả thuyết cần khảo sát thêm, chưa được chứng minh là nguyên nhân duy nhất.

Oracle tính lùi bằng simulator đã biết chỉ dùng đối chiếu kỳ vọng (B12). Nó không
cấp action/Q tối ưu làm nhãn hoặc khởi tạo learner; không gọi là tác tử model-free
đã học. Không dùng kết quả âm để âm thầm tăng budget hoặc tune lại. Một nghiên cứu
tiếp theo có thể đăng ký trước budget/ablation mới, rồi thu bộ đánh giá mới để
kiểm giả thuyết, thay vì sử dụng lại các score cuối hiện tại.



In [148]:
if RUN_EXPERIMENTS:
    after = {row["path"]: sha(BASE / row["path"]) for row in inputs}
    assert before == after
    print("PASS: all", len(after), "input files unchanged. Training steps=0; final forwards=0.")

In [149]:
import inspect
_notebook_defined = [(name, key) for name, mod in INLINE_MODULES.items() for key, obj in vars(mod).items() if inspect.isfunction(obj) and obj.__module__ == name]
assert all(vars(INLINE_MODULES[name])[key].__code__.co_filename.startswith("<notebook>/") for name,key in _notebook_defined)
print({"module có code ngay trong notebook": sum(bool(getattr(m,"__notebook_source__",False)) for m in INLINE_MODULES.values()), "hàm inline":len(_notebook_defined), "train/TEST đã chạy":False})


{'module có code ngay trong notebook': 22, 'hàm inline': 67, 'train/TEST đã chạy': False}
